# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
from sklearn.metrics import f1_score, roc_auc_score, recall_score, precision_score, accuracy_score, roc_curve
import tqdm
import scipy.stats as stats

import warnings
warnings.filterwarnings('ignore')

# User-Defined Parameteres

In [ ]:
DATAPATH1 = 'ACE-AI/Data/Raw Data/ACE_Data_2020/'
DATAPATH2 = 'ACE-AI/Data/Raw Data/Training_Data_All/'
DATAPATH3 = 'ACE-AI/Data/Raw Data/'
PROCDATAPATH = 'ACE-AI/Data/Processed Data/'
SUBDATAPATH = PROCDATAPATH + 'subdata_fixedstep/'
SUBDATAPATH2 = PROCDATAPATH + 'subdata/'
FIGPATH = 'ACE-AI/Figtbl/Sub Figures/'
MODELPATH = 'ACE-AI/Model/'

DATAFILE_2019 = 'DP_Phase4_2019_ALL.csv'
DATAFILE_TRAIN = 'DP_Phase4_combined_v3_train.csv'
DATAFILE_TEST = 'DP_Phase4_combined_v3_test.csv'
DATAFILE_2020 = 'ACE_FEATURES_ALL.csv'
NEWDATAFILE = 'DP_Phase4_combined_v3_newdata_wpreds.csv'
RISKBANDTHRES_FILE = 'ACE_Target_RiskBands.csv'

In [ ]:
target_col = [
    'Cerebral_Vascular_Accident', 'Diabetes_Mellitus',
    'Dyslipidaemia', 'Heart_and_Circulatory', 'Hypertension',
    'Osteoporosis', 'Renal_Disease', 'Osteoarthritis', 'Deceased'
]
demographics_col = ['GENDER_Male', 'GENDER_Female', 'RACE_Chinese', 'RACE_Malay', 'RACE_Indian', 'RACE_Others', 'PAT_AGE']
target_prior_col = [target+'_Prior' for target in target_col[:-1]]
target_proba_col = [target+'_Proba' for target in target_col]

med_col= [target+'_Medication' for target in target_col[:-1]]
lab_col = [target+'_Test' for target in target_col[:-1]]
diag_col = [target+'_Diagnosed' for target in target_col[:-1]]

# features
features_tbl = ['PAT_ID'] + demographics_col + target_col + \
  target_prior_col + med_col + lab_col + diag_col

# Table 1. Descriptive Cohort Stats

## Load Data

In [ ]:
%%time
orgdata_2019 = pd.read_csv(DATAPATH2 + DATAFILE_2019, usecols=features_tbl)
orgdata_train = pd.read_csv(DATAPATH2 + DATAFILE_TRAIN, usecols=features_tbl)
orgdata_test = pd.read_csv(DATAPATH2 + DATAFILE_TEST, usecols=features_tbl)

In [ ]:
%%time
data_test = orgdata_test.copy()
data_2018_all = pd.concat([orgdata_train, orgdata_test], ignore_index=True)
data_all = pd.concat([data_2018_all, orgdata_2019[
    ~orgdata_2019['PAT_ID'].str[:-4].isin(data_2018_all['PAT_ID'])]], ignore_index=True)

Assumption:
* Unique at patient level
* Prevalence rate and existing rate: Use the status of 2018 cohort to do, unless the patients only exist in 2019 cohort

## Demographics

In [ ]:
def generate_demographics(df):

    demographics_df = pd.DataFrame()

    # Gender and Race
    for col in ['GENDER_Male', 'GENDER_Female', 'RACE_Chinese', 'RACE_Malay', 'RACE_Indian', 'RACE_Others']:
        tmp = pd.DataFrame({'Demographics': col,
                            'Count': len(df[df[col]==1])}, index=[0])
        demographics_df = pd.concat([demographics_df, tmp], ignore_index=True)

    # Age
    tmp = pd.DataFrame({'Demographics': '18-39',
                            'Count': len(df[df['PAT_AGE']<=39])}, index=[0])
    demographics_df = pd.concat([demographics_df, tmp], ignore_index=True)

    tmp = pd.DataFrame({'Demographics': '40-69',
                            'Count': len(df[(df['PAT_AGE']>39)&(df['PAT_AGE']<=69)])}, index=[0])
    demographics_df = pd.concat([demographics_df, tmp], ignore_index=True)

    tmp = pd.DataFrame({'Demographics': '70+',
                            'Count': len(df[(df['PAT_AGE']>69)])}, index=[0])
    demographics_df = pd.concat([demographics_df, tmp], ignore_index=True)

    # Combine
    demographics_df['Total'] = len(df)
    demographics_df['Percent'] = demographics_df['Count'] / demographics_df['Total']

    # Add mean +/- sd of age
    demographics_df.loc[len(demographics_df)] = {'Demographics': 'Mean',
                                                 'Count': round(np.mean(df['PAT_AGE'].values), 1)}
    demographics_df.loc[len(demographics_df)] = {'Demographics': 'SD',
                                                 'Count': round(np.std(df['PAT_AGE'].values), 1)}
    demographics_df.loc[len(demographics_df)] = {'Demographics': 'No. Patients',
                                                 'Count': df['PAT_ID'].nunique()}

    return demographics_df

In [ ]:
demo_test = generate_demographics(data_test)
demo_train = generate_demographics(data_all[~data_all['PAT_ID'].isin(data_test['PAT_ID'])])
demo_all = generate_demographics(data_all)

## Prevalence Rate

In [ ]:
def generate_target_prevalence(df):

    prevalence_df = pd.DataFrame()
    for col in target_prior_col:
        tmp = pd.DataFrame({'Target': col.replace('_Prior', ''),
                            'Count': len(df[df[col]==1])}, index=[0])
        prevalence_df = pd.concat([prevalence_df, tmp], ignore_index=True)

    prevalence_df['Total'] = len(df)
    prevalence_df['Percent'] = prevalence_df['Count'] / prevalence_df['Total']

    return prevalence_df

In [ ]:
prevalence_test = generate_target_prevalence(data_test)
prevalence_train = generate_target_prevalence(data_all[~data_all['PAT_ID'].isin(data_test['PAT_ID'])])
prevalence_all = generate_target_prevalence(data_all)

### Uncoded Diagnosis

In [ ]:
def generate_target_uncoded(df):

    uncoded_df = pd.DataFrame()
    for col in target_col[:-1]:
        tmp = pd.DataFrame({'Target': col,
                            'Count': len(df[(df[col + '_Prior']>0)&(df[col + '_Diagnosed']==0)])}, index=[0])
        uncoded_df = pd.concat([uncoded_df, tmp], ignore_index=True)

    return uncoded_df

In [ ]:
uncoded_test = generate_target_uncoded(data_test)
uncoded_train = generate_target_uncoded(data_all[~data_all['PAT_ID'].isin(data_test['PAT_ID'])])
uncoded_all = generate_target_uncoded(data_all)

## Existing

In [ ]:
def generate_existing_conditions(df):

    conditions = {'No Existing': df[target_prior_col].sum(axis=1)==0,
                  '1 Existing': df[target_prior_col].sum(axis=1)==1,
                  '2+ Existing': df[target_prior_col].sum(axis=1)>1}

    existing_df = pd.DataFrame()
    for key, val in conditions.items():
        tmp = pd.DataFrame({'Condition': key,
                            'Count': len(df[val])}, index=[0])
        existing_df = pd.concat([existing_df, tmp], ignore_index=True)

    existing_df['Total'] = len(df)
    existing_df['Percent'] = existing_df['Count'] / existing_df['Total']

    return existing_df

In [ ]:
exist_test = generate_existing_conditions(data_test)
exist_train = generate_existing_conditions(data_all[~data_all['PAT_ID'].isin(data_test['PAT_ID'])])
exist_all = generate_existing_conditions(data_all)

## Combine

In [ ]:
tmptbl1 = pd.merge(demo_all, demo_train, on=['Demographics'])
tmptbl1 = pd.merge(tmptbl1, demo_test, on=['Demographics'])
tmptbl1 = tmptbl1.rename(columns={'Demographics': 'Items'})
tmptbl2 = pd.merge(prevalence_all, prevalence_train, on=['Target'])
tmptbl2 = pd.merge(tmptbl2, prevalence_test, on=['Target'])
tmptbl2 = tmptbl2.rename(columns={'Target': 'Items'})
tmptbl3 = pd.merge(exist_all, exist_train, on=['Condition'])
tmptbl3 = pd.merge(tmptbl3, exist_test, on=['Condition'])
tmptbl3 = tmptbl3.rename(columns={'Condition': 'Items'})
tbl = pd.concat([tmptbl1, tmptbl2, tmptbl3], ignore_index=True)

# Add uncoded diagnosis
tmptbl4 = pd.merge(uncoded_all, uncoded_train, on=['Target'])
tmptbl4 = pd.merge(tmptbl4, uncoded_test, on=['Target'])
tmptbl4 = tmptbl4[['Target', 'Count_x', 'Count_y', 'Count']]
tmptbl4.columns = ['Items', 'Uncoded_All', 'Uncoded_Train', 'Uncoded_Test']
tbl = pd.merge(tbl, tmptbl4, on='Items', how='left')

# Reorganize
tbl.loc[len(tbl)] = {'Items': 'Gender, n (%)'}
tbl.loc[len(tbl)] = {'Items': 'Ethnicity, n (%)'}
tbl.loc[len(tbl)] = {'Items': 'Age in Years'}
tbl.loc[len(tbl)] = {'Items': 'Pre-existing Disease at Index Date, n (%, uncoded %)'}
tbl.loc[len(tbl)] = {'Items': 'No. Pre-existing Diseases at Index Date, n (%)'}

tbl = tbl.loc[[11,23,1,0, 24, 2,3,4,5,
               25, 9, 10, 6,7,8,
               26, 13, 14, 16, 12, 15, 18, 17, 19, 27, 20, 21, 22]].reset_index(drop=True)
tbl['Items'] = ['No. Patients', 'Gender, n (%)', 'Female', 'Male',
       'Ethnicity, n (%)', 'Chinese', 'Malay', 'Indian', 'Others',
       'Age in Years', 'Mean', 'SD', '18-39', '40-69', '70+',
       'Disease Identified at the Prediction, n (%, % potentially uncoded in disease identified)',
       'Diabetes Mellitus', 'Dyslipidaemia', 'Hypertension', 'Stroke', 'Heart Disease', 'Renal Disease',
       'Osteoporosis', 'Osteoarthritis', 'No. Diseases Identified at the Prediction, n (%)', 'No Target Disease',
       '1 Target Disease', '≥ 2 Target Diseases']

# # Uncoded situation
tbl['Uncoded_Per'] = tbl['Uncoded_All']/tbl[tbl['Items']=='No. Patients'][
    'Count_x'].values[0]
tbl['Train_Uncoded_Per'] = tbl['Uncoded_Train']/tbl[tbl['Items']=='No. Patients'][
    'Count_y'].values[0]
tbl['Test_Uncoded_Per'] = tbl['Uncoded_Test']/tbl[tbl['Items']=='No. Patients'][
    'Count'].values[0]

tbl.drop(columns=['Total_x', 'Total_y', 'Total'], inplace=True)

tbl = tbl[['Items', 'Count_x', 'Percent_x', 'Uncoded_All', 'Uncoded_Per',
           'Count_y', 'Percent_y', 'Uncoded_Train', 'Train_Uncoded_Per',
           'Count', 'Percent',  'Uncoded_Test', 'Test_Uncoded_Per']]
tbl.columns = ['Items', 'Overall_Patients', 'Overall_Per', 'Overall_Uncoded',
               'Overall_Uncoded_Per', 'Train_Patients', 'Train_Per',
               'Train_Uncoded', 'Train_Uncoded_Per',
               'Test_Patients', 'Test_Per', 'Test_Uncoded', 'Test_Uncoded_Per']

# Save
tbl.to_csv(PROCDATAPATH + 'descriptive_stats.csv')

# Table 2. Model Performance with 95% CI (Fixed Steps)

In [ ]:
TESTPREDSFILE = 'DP_Phase4_combined_v3_test_wpreds.csv'
TRAINPREDSFILE = 'DP_Phase4_combined_v3_train_wpreds.csv'

In [ ]:
test_wpreds = pd.read_csv(PROCDATAPATH + TESTPREDSFILE)

## Specific-Defined Parameters

In [ ]:
N_BOOTSTRAPS    = 1000    # number of bootstrap iterations
SEED            = 42      # random seed for reproducibility
CI_LOWER        = 2.5     # lower percentile for 95% CI
CI_UPPER        = 97.5    # upper percentile for 95% CI
THRESHOLD_STEP  = 0.01    # step size for threshold search

# Sample size rule (per supervisor instruction):
#   subgroup size > 10,000 → sample 10,000 per iteration
#   subgroup size ≤ 10,000 → sample  5,000 per iteration
SAMPLE_SIZE_LARGE = 10000
SAMPLE_SIZE_SMALL =  5000
SAMPLE_SIZE_THRESHOLD = 10000

In [ ]:
target_order = [
    'Diabetes_Mellitus',
    'Dyslipidaemia',
    'Hypertension',
    'Cerebral_Vascular_Accident',
    'Heart_and_Circulatory',
    'Renal_Disease',
    'Osteoporosis',
    'Osteoarthritis'
]

In [ ]:
subgroups = [
    {
        'name':   'Overall',
        'filter': lambda df: df,
        'optimal_thresholds': dict(),
    },
    # --- Gender ---
    {
        'name':   'Male',
        'filter': lambda df: df[df['GENDER_Male'] == 1],
        'optimal_thresholds': dict(),
    },
    {
        'name':   'Female',
        'filter': lambda df: df[df['GENDER_Female'] == 1],
        'optimal_thresholds': dict(),
    },
    # --- Race ---
    {
        'name':   'Chinese',
        'filter': lambda df: df[df['RACE_Chinese'] == 1],
        'optimal_thresholds': dict(),
    },
    {
        'name':   'Malay',
        'filter': lambda df: df[df['RACE_Malay'] == 1],
        'optimal_thresholds': dict(),
    },
    {
        'name':   'Indian',
        'filter': lambda df: df[df['RACE_Indian'] == 1],
        'optimal_thresholds': dict(),
    },
    {
        'name':   'Others',
        'filter': lambda df: df[df['RACE_Others'] == 1],
        'optimal_thresholds': dict(),
    },
]

## Best Performance

In [ ]:
def create_target_df(target_prior_col, target_col, labels_data, predictions_data, data):
    df = data[target_prior_col].copy()
    df['Deceased_Prior'] = 0
    for i, target in enumerate(target_col):
        df[target] = labels_data[:, i]
        df[target + '_Proba'] = predictions_data[:, i]
    return df


# create dataframe with all combinations of f1
def create_f1_df(target_prior_col, target_proba_col, target_col,
                  labels_data, predictions_data, data, remove_prior=False):
    target_df = create_target_df(target_prior_col, target_col, labels_data, predictions_data, data)

    f1_df = pd.DataFrame()
    total_samples = len(target_df)
    range_i = np.linspace(0.1, 0.6, 51)
    for j, target in tqdm.tqdm(enumerate(target_col)):
        print(f'create f1 for: {target}')
        if remove_prior:
            df = target_df[target_df[target + '_Prior'] == 0]
        else:
            df = target_df.copy()
            df[target] = np.where(df[target + '_Prior'] == 1, 1, df[target])

        tmp = pd.DataFrame({
            'Target': [target for i in range_i],
            'ROC-AUC': [roc_auc_score(df[target], df[target_proba_col[j]]) for i in range_i],
            'Threshold': [i for i in range_i],
            'F1': [f1_score(df[target], df[target_proba_col[j]] > i) for i in range_i],
            'Precision': [precision_score(df[target], df[target_proba_col[j]] > i) for i in range_i],
            'Recall': [recall_score(df[target], df[target_proba_col[j]] > i) for i in range_i],
            'Specificity': [recall_score(df[target], df[target_proba_col[j]] > i, pos_label=0) for i in range_i],
            'Accuracy': [accuracy_score(df[target], df[target_proba_col[j]] > i) for i in range_i],
            'Test_samples': [len(df) for i in range_i],
            'Test_positive': [len(df[df[target] == 1]) for i in range_i],
            'Total_samples': [total_samples for i in range_i],
        })
        f1_df = pd.concat([f1_df, tmp], ignore_index=True)

    return f1_df

In [ ]:
# create dataframe with max f1 and metrics
def create_metrics_df(target_prior_col, target_proba_col, target_col,
                       labels_data, predictions_data, data, remove_prior=False):
    f1_df = create_f1_df(target_prior_col, target_proba_col, target_col,
                          labels_data, predictions_data, data, remove_prior)

    max_f1_df = pd.DataFrame(columns=[
        'Target', 'ROC-AUC', 'Threshold', 'F1', 'Precision', 'Recall',
        'Specificity', 'Accuracy', 'Test_samples', 'Test_positive'
    ])

    for target in tqdm.tqdm(target_col):
        print(f'create metrics for {target}')
        df = f1_df[f1_df['Target'] == target].copy()
        ind = df['F1'].idxmax()
        tmp = pd.DataFrame({
            'Target': [target],
            'ROC-AUC': [df.loc[ind, 'ROC-AUC']],
            'Threshold': [df.loc[ind, 'Threshold']],
            'F1': [df.loc[ind, 'F1']],
            'Precision': [df.loc[ind, 'Precision']],
            'Recall': [df.loc[ind, 'Recall']],
            'Specificity': [df.loc[ind, 'Specificity']],
            'Accuracy': [df.loc[ind, 'Accuracy']],
            'Test_samples': [df.loc[ind, 'Test_samples']],
            'Test_positive': [df.loc[ind, 'Test_positive']],
            'Total_samples': [df.loc[ind, 'Total_samples']]
        })
        max_f1_df = pd.concat([max_f1_df, tmp], ignore_index=True)

    return max_f1_df


def evaluate_model_from_scores(data, labels_data, predictions_data, remove_prior=False, output_all=False):
    max_f1 = create_metrics_df(target_prior_col, target_proba_col, target_col,
                                labels_data, predictions_data, data, remove_prior)
    max_f1 = max_f1[max_f1['Target'] != 'Deceased']
    if output_all:
        return max_f1, predictions_data
    else:
        return max_f1

### Run for All

In [ ]:
delong_rows = []
best_tbl = pd.DataFrame()

for sg in subgroups:
    name  = sg['name']
    sg_df = sg['filter'](test_wpreds).reset_index(drop=True)
    print(f'---------{name}------------------')

    test_max_f1 = evaluate_model_from_scores(sg_df, sg_df[target_col].values,
                                             sg_df[target_proba_col].values, remove_prior=True)
    tbl = test_max_f1.iloc[[1,2,4,0,3,6,5,7]]
    tbl['Subgroup'] = name
    best_tbl = pd.concat([best_tbl, tbl], ignore_index=True)

best_tbl.to_csv(f'{SUBDATAPATH}/best_performance.csv', index=False)

print('Saved: best_performance for all subgroups')


## Bootstrap CI

1. **Sample** patients from the group **with replacement**
  - **10,000** patients if group size > 5,000
  - **5,000** patients if group size ≤ 5,000
2. **Filter priors** — for each condition, remove patients with a prior diagnosis (`{condition}_Prior == 1`).
3. **Skip degenerate samples** — skip if only one class is present after filtering.
4. **Find best threshold** — search 0.0 to 0.5 in 0.01 steps (51 candidates) to maximise F1. **Compute F1, Precision, Recall, Accuracy** at the best threshold.
5. The **95% CI** is the 2.5th–97.5th percentile of the 1,000 collected values per metric.

In [ ]:
def run_bootstrap(subgroup_df, target_col, target_prior_col, target_proba_col, sample_size,
                  n_bootstraps=N_BOOTSTRAPS, seed=SEED):
    """
    Run bootstrap CI estimation on a pre-filtered subgroup dataframe.
    Sample size is determined automatically based on subgroup size.
    """
    n = len(subgroup_df)
    rng = np.random.default_rng(seed)

    results = {
        target: {'AUC': [], 'F1': [], 'Precision': [], 'Recall': [], 'Specificity': [], 'Accuracy': [], 'Best_Threshold': []}
        for target in target_col
    }
    prior_removal_ratio = {target: [] for target in target_col}

    # Fixed threshold grid used to search for the F1-maximising cutoff
    range_i = np.linspace(0.1, 0.6, 51)

    for i in tqdm.tqdm(range(n_bootstraps)):

        # Step 1 & 2: Sample from subgroup with replacement
        idx    = rng.choice(n, size=sample_size, replace=True)
        sample = subgroup_df.iloc[idx]

        for j, target in enumerate(target_col):
            proba_col = target_proba_col[j]
            prior_col = target_prior_col[j]

            # Step 3: Filter out patients with prior diagnosis
            df = sample[sample[prior_col] == 0]
            prior_removal_ratio[target].append((sample_size - len(df)) / sample_size)

            y_true  = df[target].values
            y_score = df[proba_col].values

            # Step 4: Compute AUC
            results[target]['AUC'].append(roc_auc_score(y_true, y_score))

            # Steps 5: Grid search over thresholds to find max F1
            best_f1, best_prec, best_rec, best_spec, best_acc, best_thresh = 0.0, 0.0, 0.0, 0.0, 0.0, 0.0

            f1_scores  = np.array([f1_score(y_true, y_score > t, zero_division=0) for t in range_i])
            precisions = np.array([precision_score(y_true, y_score > t, zero_division=0) for t in range_i])
            recalls    = np.array([recall_score(y_true, y_score > t, zero_division=0) for t in range_i])

            best_idx    = np.argmax(f1_scores)
            best_f1     = f1_scores[best_idx]
            best_prec   = precisions[best_idx]
            best_rec    = recalls[best_idx]
            best_thresh = range_i[best_idx]

            # For Specificity/Accuracy use prediction at best threshold
            y_pred    = (y_score > best_thresh).astype(int)
            best_spec = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
            best_acc  = accuracy_score(y_true, y_pred)

            results[target]['F1'].append(best_f1)
            results[target]['Precision'].append(best_prec)
            results[target]['Recall'].append(best_rec)
            results[target]['Specificity'].append(best_spec)
            results[target]['Accuracy'].append(best_acc)
            results[target]['Best_Threshold'].append(best_thresh)

    gc.collect()
    return results, prior_removal_ratio


def build_ci_df(results, target_col, subgroup_name, subgroup_n, sample_size):
    """Summarise bootstrap results into a CI table."""
    ci_rows = []
    for target in target_col:
        row = {'Subgroup': subgroup_name, 'Subgroup_N': subgroup_n,
               'Sample_size': sample_size, 'Target': target}
        for metric in ['AUC', 'F1', 'Precision', 'Recall', 'Specificity', 'Accuracy']:
            vals = np.array(results[target][metric])
            row[f'{metric}_mean']     = np.mean(vals)
            row[f'{metric}_CI_lower'] = np.percentile(vals, CI_LOWER)
            row[f'{metric}_CI_upper'] = np.percentile(vals, CI_UPPER)
        thresh_vals = np.array(results[target]['Best_Threshold'])
        row['Threshold_mean'] = np.mean(thresh_vals)
        row['Threshold_std']  = np.std(thresh_vals)
        row['Threshold_min']  = np.min(thresh_vals)
        row['Threshold_max']  = np.max(thresh_vals)
        ci_rows.append(row)

    return pd.DataFrame(ci_rows, columns=[
        'Subgroup', 'Subgroup_N', 'Sample_size', 'Target',
        'AUC_mean',         'AUC_CI_lower',         'AUC_CI_upper',
        'F1_mean',          'F1_CI_lower',           'F1_CI_upper',
        'Precision_mean',   'Precision_CI_lower',    'Precision_CI_upper',
        'Recall_mean',      'Recall_CI_lower',       'Recall_CI_upper',
        'Specificity_mean', 'Specificity_CI_lower',  'Specificity_CI_upper',
        'Accuracy_mean',    'Accuracy_CI_lower',     'Accuracy_CI_upper',
        'Threshold_mean',   'Threshold_std',         'Threshold_min', 'Threshold_max',
    ])

### Run for All

In [ ]:
all_ci = []  # collect all subgroup CI results for combined output

for sg in subgroups:
    name = sg['name']
    
    print(f'\n=== {name} ===')

    # Filter to subgroup
    sg_df = sg['filter'](test_wpreds).reset_index(drop=True)
    n     = len(sg_df)
    if name in ['Chinese', 'Malay', 'Indian', 'Others']:  # Most ethnicities < 10K patients
        sample_size = SAMPLE_SIZE_SMALL
    else:
        sample_size = SAMPLE_SIZE_LARGE

    # Run bootstrap
    results, prior_removal_ratio = run_bootstrap(
        sg_df, target_col[:-1], target_prior_col, target_proba_col,
        sample_size
    )

    # Build CI dataframe
    ci_df = build_ci_df(results, target_col[:-1], name, n, sample_size)
    all_ci.append(ci_df)

    # Save per-subgroup results
    safe_name = name.replace(' ', '_').replace('/', '').replace('(', '').replace(')', '').replace('>', 'gte').replace('<', 'lt')
    ci_df.to_csv(f'{SUBDATAPATH}/bootstrap_CI_{safe_name}.csv', index=False)
    pd.DataFrame(prior_removal_ratio).to_csv(
        f'{SUBDATAPATH}/prior_removal_ratio_{safe_name}.csv', index=False
    )
    print(f'Saved: bootstrap_CI_{safe_name}.csv')

## Delong ROC-AUC CI

In [ ]:
def compute_midrank(x):
    """
    Computes midranks of a 1D array.
    Equivalent to R's rank(x, ties.method='average').
    """
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=np.float64)
    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        # Calculate the average rank for this tie-group
        T[i:j] = (i + j + 1) / 2.0
        i = j

    out = np.empty(N, dtype=np.float64)
    out[J] = T
    return out

def delong_auc_ci(y_true, y_score, alpha=0.05):
    """
    Rewritten to match pAUC/pROC output using Sun & Xu (2014) logic.
    """
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    # Separate scores by class
    pos = y_score[y_true == 1]
    neg = y_score[y_true == 0]
    m, n = len(pos), len(neg)

    # 1. Ranks of all scores combined
    combined_scores = np.concatenate([pos, neg])
    combined_ranks = compute_midrank(combined_scores)

    # 2. Ranks within each group separately
    pos_ranks_internal = compute_midrank(pos)
    neg_ranks_internal = compute_midrank(neg)

    # 3. Structural components (V10 and V01)
    # V10[i] is based on ranks of positive cases relative to negative cases
    v10 = (combined_ranks[:m] - pos_ranks_internal) / n
    # V01[j] is based on ranks of negative cases relative to positive cases
    v01 = (m + 1 - (combined_ranks[m:] - neg_ranks_internal)) / m

    # Calculate AUC and Variance
    auc = np.mean(v10)
    var_auc = (np.var(v10, ddof=1) / m) + (np.var(v01, ddof=1) / n)

    # 4. Normal-approximation Confidence Interval
    se = np.sqrt(var_auc)
    z = stats.norm.ppf(1 - alpha / 2)

    lower = max(0.0, auc - z * se)
    upper = min(1.0, auc + z * se)

    return auc, (lower, upper)


### Run for All

In [ ]:
delong_rows = []

for sg in subgroups:
    name  = sg['name']
    sg_df = sg['filter'](test_wpreds).reset_index(drop=True)

    for j, target in enumerate(target_col[:-1]):
        prior_col = target_prior_col[j]
        proba_col = target_proba_col[j]

        # Filter out prior patients (same as bootstrap)
        df = sg_df[sg_df[prior_col] == 0]

        y_true  = df[target].values
        y_score = df[proba_col].values

        if len(np.unique(y_true)) < 2:
            continue

        auc, (ci_lower, ci_upper) = delong_auc_ci(y_true, y_score)

        delong_rows.append({
            'Subgroup':   name,
            'Target':  target,
            'N':          len(df),
            'AUC':        round(auc,      6),
            'CI_Lower':   round(ci_lower, 6),
            'CI_Upper':   round(ci_upper, 6),
            'CI_Width':   round(ci_upper - ci_lower, 6),
        })

delong_df = pd.DataFrame(delong_rows)

# Set ordered categorical
delong_df['Target'] = pd.Categorical(
    delong_df['Target'],
    categories=target_order,
    ordered=True
)

# Sort by Subgroup first, then Condition order
delong_df = delong_df.sort_values(
    ['Subgroup', 'Target']
).reset_index(drop=True)

delong_df.to_csv(
    f'{SUBDATAPATH}/delong_AUC_CI.csv',
    index=False
)

print('Saved: delong_AUC_CI.csv for all subgroups')


## Combine

1. Use Bootstrap CI for all metrics except for ROC-AUC
2. Use Delong ROC-AUC CIs
3. Insert best performance metrics together with the corresponding CIs

In [ ]:
files = [i for i in os.listdir(SUBDATAPATH) if 'bootstrap_CI_' in i]
all_ci = []
for f in files:
    all_ci.append(pd.read_csv(SUBDATAPATH + f))
combined_ci = pd.concat(all_ci, ignore_index=True)

# Step 2. Set ordered categorical
combined_ci['Target'] = pd.Categorical(
    combined_ci['Target'],
    categories=target_order,
    ordered=True
)

# Step 3. Sort by Subgroup first, then Condition order
combined_ci = combined_ci.sort_values(
    ['Subgroup', 'Target']
).reset_index(drop=True)

# Step 4. Load Delong ROC-AUC CIs and best performance metrics
delong_ci = pd.read_csv(SUBDATAPATH + 'delong_AUC_CI.csv')
delong_ci.columns = [i + '_delong' if i in ['N', 'AUC', 'CI_Lower', 'CI_Upper', 'CI_Width'] else i for i in delong_ci.columns]
best_perf = pd.read_csv(SUBDATAPATH + 'best_performance.csv')
best_perf.rename(columns={'ROC-AUC':'AUC'}, inplace=True)
best_perf.columns = [i + '_bestperf' if i in ['AUC', 'Threshold', 'F1', 'Precision', 'Recall',
        'Specificity', 'Accuracy', 'Test_samples', 'Test_positive', 'Total_samples'] else i for i in best_perf.columns]

# Step 5: Merge additional columns from delong_ci and best_perf, matched on Subgroup + Target
print(f'Before merge: size {combined_ci.shape}')
combined_ci = combined_ci.merge(delong_ci, on=['Subgroup', 'Target'], how='left')
combined_ci = combined_ci.merge(best_perf, on=['Subgroup', 'Target'], how='left')
print(f'After merge: size {combined_ci.shape}')

# Step 6: Formatting (with 5 empty rows after each subgroup)
empty_rows = pd.DataFrame([{col: pd.NA for col in combined_ci.columns}] * 5)
dfs = []
for i, subgroup in enumerate(combined_ci['Subgroup'].drop_duplicates()):
    dfs.append(combined_ci[combined_ci['Subgroup'] == subgroup])

    # Add empty rows except after the last subgroup
    if i < len(subgroups) - 1:
        dfs.append(empty_rows.copy())
combined_ci_spaced = pd.concat(dfs, ignore_index=True)

# Step 7: Save
combined_ci.to_csv(f'{SUBDATAPATH}/Performance_CI_all_subgroups_allinfo_nospace.csv', index=False)
combined_ci_spaced.to_csv(f'{SUBDATAPATH}/Performance_CI_all_subgroups_allinfo.csv', index=False)

### Even Nicer Formatting

In [ ]:
def format_metric(row, best_col, lower_col, upper_col, decimals=3):
    """Format as 'value (lower-upper)' string."""
    val   = row[best_col]
    lower = row[lower_col]
    upper = row[upper_col]
    if pd.isna(val):
        return ''
    return f"{val:.{decimals}}\n ({lower:.{decimals}f}-{upper:.{decimals}f})"


def build_formatted_table(df, subgroup_name):
    """Build a formatted performance table for a subgroup"""
    sub = df[df['Subgroup'] == subgroup_name].copy()

    out = pd.DataFrame()
    out['Target Disease'] = sub['Target']

    # ROC-AUC: bestperf value, but CI from delong
    out['AUC'] = sub.apply(lambda r: format_metric(
        r, 'AUC_bestperf', 'CI_Lower_delong', 'CI_Upper_delong'), axis=1)

    # Max F1: bestperf value, CI from bootstrap F1 CI
    out['Max F1'] = sub.apply(lambda r: format_metric(
        r, 'F1_bestperf', 'F1_CI_lower', 'F1_CI_upper'), axis=1)

    # Precision
    out['Precision'] = sub.apply(lambda r: format_metric(
        r, 'Precision_bestperf', 'Precision_CI_lower', 'Precision_CI_upper'), axis=1)

    # Recall
    out['Recall'] = sub.apply(lambda r: format_metric(
        r, 'Recall_bestperf', 'Recall_CI_lower', 'Recall_CI_upper'), axis=1)

    # Accuracy
    out['Accuracy'] = sub.apply(lambda r: format_metric(
        r, 'Accuracy_bestperf', 'Accuracy_CI_lower', 'Accuracy_CI_upper'), axis=1)

    # Simple Average row (mean of point estimates only, no CI)
    avg_row = {
        'Target Disease': 'Simple Average',
        'AUC':    f"{sub['AUC_bestperf'].mean():.3f}",
        'Max F1':     f"{sub['F1_bestperf'].mean():.3f}",
        'Precision': f"{sub['Precision_bestperf'].mean():.3f}",
        'Recall':    f"{sub['Recall_bestperf'].mean():.3f}",
        'Accuracy':   f"{sub['Accuracy_bestperf'].mean():.3f}",
    }
    out = pd.concat([out, pd.DataFrame([avg_row])], ignore_index=True)

    return out

In [ ]:
all_formatted = []

for i, subgroup in enumerate(combined_ci['Subgroup'].drop_duplicates()):
    formatted = build_formatted_table(combined_ci, subgroup)
    formatted.insert(0, 'Subgroup', subgroup)  # keep track of which subgroup each block belongs to

    # Format target diseases
    formatted['Target Disease'] = ['Diabetes Mellitus', 'Dyslipidaemia', 'Hypertension', 'Stroke',
                                   'Heart Disease', 'Renal Disease', 'Osteoporosis', 'Osteoarthritis',
                                   'Simple Average']
    all_formatted.append(formatted)
    all_formatted.append(pd.DataFrame([{col: pd.NA for col in formatted.columns}] * 5))

# Combined CSV with all subgroups stacked, each block labeled by Subgroup
all_formatted_df = pd.concat(all_formatted, ignore_index=True)
all_formatted_df.to_excel(
    f'{PROCDATAPATH}/performance_CI_niceformat.xlsx', index=False)

print('Saved per-subgroup CSVs and combined formatted xlsx.')

# Fig 1. Precision @K Curves

In [ ]:
TESTPREDSFILE = 'DP_Phase4_combined_v3_test_wpreds.csv'
test_wpreds = pd.read_csv(PROCDATAPATH + TESTPREDSFILE)

In [ ]:
def calculate_precision_at_k(df, true_col, proba_col, top_fracs=None):
    """Precision (PPV), lift and NNS at user-specified top-k cohort sizes.

    top_fracs: iterable of top proportions, as fractions ([0.005, 0.01, 0.05])
        or percentages ([0.5, 1, 5]). Percentages are detected when any value
        exceeds 1. Defaults to [0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.50, 1.0].
    """

    # 1. Validation
    if top_fracs is None:
        top_fracs = [0.005, 0.01, 0.02, 0.05, 0.10, 0.20, 0.50, 1.0]
    top_fracs = np.asarray(list(top_fracs), dtype=float)

    if top_fracs.size == 0:
        raise ValueError("top_fracs is empty")
    if (top_fracs <= 0).any():
        raise ValueError(f"top_fracs must be positive, got {top_fracs.min()}")
    if top_fracs.max() > 1:                      # interpret as percentages
        raise ValueError(f"top_fracs max {top_fracs.max()} exceeds 100%")

    df_rmprior = df[df[true_col + '_Prior']==0].copy()
    sub = df_rmprior[[true_col, proba_col]]
    if sub.isna().any().any():
        n_bad = int(sub.isna().any(axis=1).sum())
        raise ValueError(f"{n_bad} rows have NaN in {true_col!r} or {proba_col!r}")

    vals = set(pd.unique(sub[true_col]))
    if not vals <= {0, 1, True, False}:
        raise ValueError(f"{true_col!r} is not binary: {sorted(vals)}")

    n = len(sub)
    if n == 0:
        raise ValueError("empty dataframe")

    # 2. Sorting
    d = sub.sort_values(proba_col, ascending=False, kind="mergesort").reset_index(drop=True)

    y = d[true_col].to_numpy(dtype=float)
    cum_tp = np.cumsum(y)
    prevalence = y.mean()
    total_pos = y.sum()

    rows, seen = [], set()
    for f in np.sort(np.unique(top_fracs)):
        k = int(min(n, max(1, np.ceil(n * f))))
        if k in seen:                            # two fracs collapsing to the same k
            continue
        seen.add(k)
        tp = cum_tp[k - 1]
        prec = tp / k
        rows.append({
            "Requested_Fraction": f,
            "K": k,
            "Population_Percentage": 100.0 * k / n,
            "True_Positives": int(tp),
            "Precision_at_K": prec,
            "Recall_at_K": tp / total_pos if total_pos > 0 else np.nan,
            "Lift": prec / prevalence if prevalence > 0 else np.nan,
            "NNS": 1.0 / prec if prec > 0 else np.inf,
        })

    out = pd.DataFrame(rows)
    out.attrs["prevalence"] = prevalence
    out.attrs["n"] = n
    out.attrs["total_positives"] = int(total_pos)
    return out


def plot_precision_at_k_chart(precision_df, disease_name, fig_path=None,
                              dataset="train", prevalence=None, logx=False,
                              annotate=True):
    """Plot a precision@k curve with a prevalence baseline.

    prevalence defaults to precision_df.attrs['prevalence'], set by
    calculate_precision_at_k on the same split the curve came from.
    """
    if prevalence is None:
        prevalence = precision_df.attrs.get("prevalence")
    if prevalence is None:
        raise ValueError("prevalence not supplied and not found in precision_df.attrs")

    x = precision_df["Population_Percentage"].to_numpy()
    yv = precision_df["Precision_at_K"].to_numpy()

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(x, yv, marker="o", linestyle="-", color="green", label="Model (PPV)")
    ax.axhline(prevalence, color="red", linestyle="--",
               label=f"Random baseline (prevalence = {prevalence:.3f})")

    if annotate:
        for xi, yi, ki in zip(x, yv, precision_df["K"]):
            ax.annotate(f"k={ki}", (xi, yi), textcoords="offset points",
                        xytext=(0, 8), ha="center", fontsize=8, alpha=0.75)

    title_txt = disease_name.replace("_", " ")
    ax.set_title(f"Precision at K for {title_txt} ({dataset.capitalize()} Data)",
                 fontsize=14, fontweight="bold")
    ax.set_xlabel("Top proportion of cohort flagged (%)", fontsize=12)
    ax.set_ylabel("Precision (PPV)", fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.7)
    ax.legend(fontsize=10)

    if logx:
        ax.set_xscale("log")
        ax.set_xlim(max(x.min() * 0.8, 1e-3), 105)
    else:
        ax.set_xlim(0, 100)

    top = max(yv.max(), prevalence)
    ax.set_ylim(0, min(1.05, top * 1.25) if top < 0.8 else 1.05)
    fig.tight_layout()

    filepath = None
    if fig_path is not None:
        os.makedirs(fig_path, exist_ok=True)
        filepath = os.path.join(fig_path, f"precision_at_k_{disease_name}_{dataset}.png")
        fig.savefig(filepath, dpi=300, bbox_inches="tight")
        print(f"Saved Precision at K chart for {disease_name} to {filepath}")
    plt.show()
    plt.close(fig)
    return filepath

## Test Set

In [ ]:
# Assuming train_wpreds is loaded and target_col, target_proba_col are defined

for i, disease in enumerate(target_col[:-1]): # Exclude 'Deceased'
    true_label_col = disease
    proba_prediction_col = target_proba_col[i] # Match index

    if true_label_col in test_wpreds.columns and proba_prediction_col in test_wpreds.columns:
        res = calculate_precision_at_k(test_wpreds, true_label_col, proba_prediction_col,
                                      top_fracs=[0.002, 0.005, 0.01, 0.15, 0.02, 0.05] + [
                                          min(i, 1) for i in np.arange(0.1, 1.05, 0.05)])
        plot_precision_at_k_chart(res, true_label_col, FIGPATH, dataset='test', logx=False,
                              annotate=False)
    else:
        print(f"Warning: Columns '{true_label_col}' or '{proba_prediction_col}' not found in test_wpreds for {disease}.")

# Fig 2. Specificity vs Ruleout Rate

In [ ]:
TESTPREDSFILE = 'DP_Phase4_combined_v3_test_wpreds.csv'
test_wpreds = pd.read_csv(PROCDATAPATH + TESTPREDSFILE)

In [ ]:
def calculate_npv_curve(df, true_col, proba_col, bottom_fracs=None):
    """NPV (and false omission rate) among the lowest-scoring fraction of the cohort.

    bottom_fracs: iterable of bottom proportions to rule out, as fractions
        ([0.10, 0.25, 0.50]) or percentages ([10, 25, 50]). Percentages are
        detected when any value exceeds 1. Defaults to
        [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99].
    """

    # 1. Validation
    if bottom_fracs is None:
        bottom_fracs = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    bottom_fracs = np.asarray(list(bottom_fracs), dtype=float)

    if bottom_fracs.size == 0:
        raise ValueError("bottom_fracs is empty")
    if (bottom_fracs <= 0).any():
        raise ValueError(f"bottom_fracs must be positive, got {bottom_fracs.min()}")
    if bottom_fracs.max() > 1:                      # interpret as percentages
        if bottom_fracs.max() > 100:
            raise ValueError(f"bottom_fracs max {bottom_fracs.max()} exceeds 100%")
        bottom_fracs = bottom_fracs / 100.0

    df_rmprior = df[df[true_col + '_Prior']==0].copy()
    sub = df_rmprior[[true_col, proba_col]]
    if sub.isna().any().any():
        n_bad = int(sub.isna().any(axis=1).sum())
        raise ValueError(f"{n_bad} rows have NaN in {true_col!r} or {proba_col!r}")

    vals = set(pd.unique(sub[true_col]))
    if not vals <= {0, 1, True, False}:
        raise ValueError(f"{true_col!r} is not binary: {sorted(vals)}")

    n = len(sub)
    if n == 0:
        raise ValueError("empty dataframe")

    # 2. Calculation
    d = sub.sort_values(proba_col, ascending=True, kind="mergesort").reset_index(drop=True)
    y = d[true_col].to_numpy(dtype=float)
    p = d[proba_col].to_numpy(dtype=float)
    cum_fn = np.cumsum(y)                            # cases inside the bottom cohort
    prevalence = y.mean()
    total_pos = y.sum()
    total_neg = n - total_pos

    rows, seen = [], set()
    for f in np.sort(np.unique(bottom_fracs)):
        m = int(min(n, max(1, np.ceil(n * f))))      # ruled-out cohort size
        if m in seen:                                # two fracs collapsing to the same m
            continue
        seen.add(m)
        fn = cum_fn[m - 1]
        tn = m - fn
        rows.append({
            "Requested_Fraction": f,
            "Ruled_Out_N": m,
            "Rule_Out_Rate": 100.0 * m / n,                       # x-axis (%)
            "Cut_Probability": p[m - 1],                          # highest score ruled out
            "NPV": tn / m,                                        # y-axis
            "False_Omission_Rate": fn / m,
            "Missed_per_1000_ruled_out": 1000.0 * fn / m,
            "False_Negatives": int(fn),
            "True_Negatives": int(tn),
            "Specificity": tn / total_neg if total_neg > 0 else np.nan,
            "Sensitivity_retained": (total_pos - fn) / total_pos if total_pos > 0 else np.nan,
        })

    out = pd.DataFrame(rows)
    out.attrs["prevalence"] = prevalence
    out.attrs["n"] = n
    out.attrs["total_positives"] = int(total_pos)
    return out


def plot_npv_curve(npv_df, disease_name, fig_path=None, dataset="test",
                   prevalence=None, metric="NPV", annotate=True):
    """Plot NPV (or false omission rate) against rule-out rate."""
    if prevalence is None:
        prevalence = npv_df.attrs.get("prevalence")
    if prevalence is None:
        raise ValueError("prevalence not supplied and not found in npv_df.attrs")

    d = npv_df.dropna(subset=[metric])
    if d.empty:
        raise ValueError(f"no rows with a defined {metric} (all thresholds rule out nobody)")

    x = d["Rule_Out_Rate"].to_numpy()
    yv = d[metric].to_numpy()
    baseline = (1.0 - prevalence) if metric == "NPV" else prevalence
    ylab = ("Negative predictive value" if metric == "NPV"
            else "False omission rate (missed cases among ruled out)")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(x, yv, marker="o", linestyle="-", color="steelblue", label=f"Model ({metric})")
    ax.axhline(baseline, color="red", linestyle="--",
               label=f"Random rule-out ({baseline:.3f})")

    if annotate:
      for xi, yi, ci in zip(x, yv, d["Cut_Probability"]):
          ax.annotate(f"p<{ci:.3g}", (xi, yi), textcoords="offset points",
                      xytext=(0, 8), ha="center", fontsize=8, alpha=0.75)

    title_txt = disease_name.replace("_", " ")
    ax.set_title(f"Rule-out performance for {title_txt} ({dataset.capitalize()} Data)",
                 fontsize=14, fontweight="bold")
    ax.set_xlabel("Proportion of cohort ruled out (%)", fontsize=12)
    ax.set_ylabel(ylab, fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.7)
    ax.legend(fontsize=10)
    ax.set_xlim(0, 100)

    lo, hi = min(yv.min(), baseline), max(yv.max(), baseline)
    pad = max(0.02 * (hi - lo), 0.002)
    ax.set_ylim(max(0, lo - pad), min(1.0, hi + pad))
    fig.tight_layout()

    filepath = None
    if fig_path is not None:
        os.makedirs(fig_path, exist_ok=True)
        suffix = "npv" if metric == "NPV" else "for"
        filepath = os.path.join(fig_path, f"ruleout_{suffix}_{disease_name}_{dataset}.png")
        fig.savefig(filepath, dpi=300, bbox_inches="tight")
        print(f"Saved rule-out curve for {disease_name} to {filepath}")
    plt.show()
    plt.close(fig)
    return filepath

## Test Set

In [ ]:
# Assuming train_wpreds is loaded and target_col, target_proba_col are defined
for i, disease in enumerate(target_col[:-1]): # Exclude 'Deceased'
    true_label_col = disease
    proba_prediction_col = target_proba_col[i] # Match index

    if true_label_col in test_wpreds.columns and proba_prediction_col in test_wpreds.columns:
        r = calculate_npv_curve(test_wpreds, true_label_col, proba_prediction_col,
                        bottom_fracs=[max(0.001, min(i, 1)) for i in np.arange(0, 1.05, 0.05)])
        plot_npv_curve(r, true_label_col, FIGPATH, dataset='test', annotate=False)                        # NPV view
    else:
        print(f"Warning: Columns '{true_label_col}' or '{proba_prediction_col}' not found in test_wpreds for {disease}.")

# Fig 3. Decision Curve Analysis

In [ ]:
TESTPREDSFILE = 'DP_Phase4_combined_v3_test_wpreds.csv'
test_wpreds = pd.read_csv(PROCDATAPATH + TESTPREDSFILE)

In [ ]:
def calculate_decision_curve(df, true_col, proba_col, thresholds=None):
    """Net benefit of the model vs treat-all and treat-none across decision thresholds.

    Restricted to the at-risk cohort (no prior diagnosis), matching
    calculate_precision_at_k.

    thresholds: iterable of decision threshold probabilities in (0, 1), e.g.
        [0.01, 0.02, 0.05, 0.10, 0.20]. The threshold encodes the clinical
        exchange rate: acting at t means one missed case is judged equivalent
        to t/(1-t) unnecessary assessments. Defaults to 0.01..0.30.
    """

    # 1. Validation
    if thresholds is None:
        thresholds = np.round(np.arange(0.01, 0.305, 0.01), 3)
    thresholds = np.asarray(list(thresholds), dtype=float)

    if thresholds.size == 0:
        raise ValueError("thresholds is empty")
    if ((thresholds <= 0) | (thresholds >= 1)).any():
        raise ValueError(f"thresholds must lie strictly in (0, 1); got "
                         f"[{thresholds.min()}, {thresholds.max()}]")

    df_rmprior = df[df[true_col + '_Prior'] == 0].copy()
    sub = df_rmprior[[true_col, proba_col]]
    if sub.isna().any().any():
        n_bad = int(sub.isna().any(axis=1).sum())
        raise ValueError(f"{n_bad} rows have NaN in {true_col!r} or {proba_col!r}")

    vals = set(pd.unique(sub[true_col]))
    if not vals <= {0, 1, True, False}:
        raise ValueError(f"{true_col!r} is not binary: {sorted(vals)}")

    n = len(sub)
    if n == 0:
        raise ValueError("empty dataframe")

    y = sub[true_col].to_numpy(dtype=float)
    p = sub[proba_col].to_numpy(dtype=float)
    prevalence = y.mean()
    total_pos = y.sum()

    rows = []
    for t in np.sort(np.unique(thresholds)):
        w = t / (1.0 - t)                        # harm of a FP, in units of a TP
        flag = p >= t
        tp = float((flag & (y == 1)).sum())
        fp = float((flag & (y == 0)).sum())
        fn = total_pos - tp

        nb_model = tp / n - (fp / n) * w
        nb_all = prevalence - (1.0 - prevalence) * w
        nb_none = 0.0
        nb_best_default = max(nb_all, nb_none)

        rows.append({
            "Threshold": t,
            "Harm_Ratio_FP_per_TP": w,
            "Flagged_N": int(flag.sum()),
            "Flagged_Pct": 100.0 * flag.sum() / n,
            "TP": int(tp), "FP": int(fp), "FN": int(fn),
            "Net_Benefit_Model": nb_model,
            "Net_Benefit_Treat_All": nb_all,
            "Net_Benefit_Treat_None": nb_none,
            "NB_per_1000": 1000.0 * nb_model,
            # net cases found per 1000 vs the better of the two default strategies
            "Net_Gain_vs_Default_per_1000": 1000.0 * (nb_model - nb_best_default),
            # assessments avoidable per 1000 at no cost in cases found
            "Net_Reduction_Assessments_per_1000":
                1000.0 * (nb_model - nb_all) / w if w > 0 else np.nan,
            "Standardized_NB": nb_model / prevalence if prevalence > 0 else np.nan,
            "Model_Best": bool(nb_model >= nb_all and nb_model >= nb_none),
        })

    out = pd.DataFrame(rows)
    out.attrs["prevalence"] = prevalence
    out.attrs["n"] = n
    out.attrs["total_positives"] = int(total_pos)
    return out


def useful_threshold_range(dc_df):
    """Contiguous threshold span over which the model beats both default strategies."""
    best = dc_df.loc[dc_df["Model_Best"], "Threshold"]
    if best.empty:
        return None
    return float(best.min()), float(best.max())


def plot_decision_curve(dc_df, disease_name, fig_path=None, dataset="test",
                        prevalence=None, standardized=False, ymin=None):
    """Plot net benefit for the model, treat-all and treat-none."""
    if prevalence is None:
        prevalence = dc_df.attrs.get("prevalence")
    if prevalence is None:
        raise ValueError("prevalence not supplied and not found in dc_df.attrs")

    x = dc_df["Threshold"].to_numpy() * 100.0
    if standardized:
        m = dc_df["Net_Benefit_Model"].to_numpy() / prevalence
        a = dc_df["Net_Benefit_Treat_All"].to_numpy() / prevalence
        ylab = "Standardized net benefit"
    else:
        m = dc_df["Net_Benefit_Model"].to_numpy()
        a = dc_df["Net_Benefit_Treat_All"].to_numpy()
        ylab = "Net benefit"

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.plot(x, m, marker="o", linestyle="-", color="darkgreen", label="ACE-AI")
    ax.plot(x, a, linestyle="-", color="grey", label="Assess all")
    ax.axhline(0.0, color="black", linestyle="--", linewidth=1, label="Assess none")

    title_txt = disease_name.replace("_", " ")
    ax.set_title(f"Decision curve for {title_txt} ({dataset.capitalize()} Data)",
                 fontsize=14, fontweight="bold")
    ax.set_xlabel("Threshold probability (%)", fontsize=12)
    ax.set_ylabel(ylab, fontsize=12)
    ax.grid(True, linestyle="--", alpha=0.7)
    ax.legend(fontsize=10)
    ax.set_xlim(x.min(), x.max())

    hi = max(m.max(), a.max(), 0.0)
    lo = ymin if ymin is not None else -0.25 * max(hi, 1e-6)
    ax.set_ylim(lo, hi * 1.15 if hi > 0 else 0.01)
    fig.tight_layout()

    filepath = None
    if fig_path is not None:
        os.makedirs(fig_path, exist_ok=True)
        suffix = "snb" if standardized else "nb"
        filepath = os.path.join(fig_path, f"decision_curve_{suffix}_{disease_name}_{dataset}.png")
        fig.savefig(filepath, dpi=300, bbox_inches="tight")
        print(f"Saved decision curve for {disease_name} to {filepath}")
    plt.show()
    plt.close(fig)
    return filepath

## Test Set

In [ ]:
# Assuming train_wpreds is loaded and target_col, target_proba_col are defined
for i, disease in enumerate(target_col[:-1]): # Exclude 'Deceased'
    true_label_col = disease
    proba_prediction_col = target_proba_col[i] # Match index

    if true_label_col in test_wpreds.columns and proba_prediction_col in test_wpreds.columns:
        r = calculate_decision_curve(test_wpreds, true_label_col, proba_prediction_col,
                        thresholds=np.arange(0.01, 0.95, 0.05))
        plot_decision_curve(r, true_label_col, FIGPATH, dataset='test')                        # NPV view
    else:
        print(f"Warning: Columns '{true_label_col}' or '{proba_prediction_col}' not found in test_wpreds for {disease}.")

# Fig 4. Race Performance with 95% CI (Fixed Step)


## AUC

In [ ]:
def plot_roc_by_race(df, output_dir=FIGPATH, disease_col="Target", race_col="Subgroup",
    truth_col="y_true", score_col="y_score"):
    """
    Generate and save one ROC plot per disease.

    Parameters
    ----------
    df : pd.DataFrame
        Long-format dataframe containing: disease, race, y_true, y_score

    output_dir : str
        Folder where plots will be saved.
    """

    diseases = sorted(df[disease_col].unique())

    # Colorblind-friendly palette
    race_styles = {"Chinese": {"color": "#0072B2", "linestyle": "-"},
        "Malay": {"color": "#E69F00", "linestyle": "--"},
        "Indian": {"color": "#009E73", "linestyle": "-."},
        "Others": {"color": "#CC79A7", "linestyle": ":"}}

    saved_files = []
    for disease in diseases:
        fig, ax = plt.subplots(figsize=(10, 8))

        disease_df = df[df[disease_col] == disease]
        for race in sorted(disease_df[race_col].unique()):
            sub = disease_df[disease_df[race_col] == race]
            y_true = sub[truth_col].values
            y_score = sub[score_col].values

            if len(np.unique(y_true)) < 2:
                continue

            fpr, tpr, _ = roc_curve(y_true, y_score)
            auc, (lower, upper) = delong_auc_ci(y_true, y_score)
            style = race_styles.get(race, {"color": "black", "linestyle": "-"})

            ax.plot(fpr, tpr, lw=2.5, color=style["color"],
                    linestyle=style["linestyle"],
                    label=(
                    f"{race}\n"
                    f"AUC={auc:.3f} "
                    f"({lower:.3f}-{upper:.3f})"
                ))

        # Reference line
        ax.plot([0, 1], [0, 1], color="grey", linestyle="--", linewidth=1, alpha=0.7)

        title_txt = disease
        if title_txt == 'Cerebral_Vascular_Accident':
            title_txt = 'Stroke'
        elif title_txt == 'Heart_and_Circulatory':
            title_txt = 'Heart Disease'
        else:
            title_txt = disease.replace("_", " ")

        ax.set_title(title_txt, fontsize=20, fontweight="bold")
        # ax.set_title(disease.replace("_", " "), fontsize=20, fontweight="bold")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)
        ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=20)
        ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=20)
        ax.grid(alpha=0.3, linestyle="--")
        ax.legend(fontsize=15, frameon=True, loc="lower right")
        ax.tick_params(axis="both", which="major", labelsize=20)
        plt.tight_layout()

        filename = disease.replace("_", " ").replace("/", "-") + ".png"
        filepath = os.path.join(output_dir, filename)
        plt.savefig(filepath, dpi=300, bbox_inches="tight")
        plt.close()

        saved_files.append(filepath)

    return saved_files

In [ ]:
delong_rows = []
roc_rows = []

for sg in subgroups:
    name  = sg['name']
    sg_df = sg['filter'](test_wpreds).reset_index(drop=True)

    for j, target in enumerate(target_col[:-1]):
        prior_col = target_prior_col[j]
        proba_col = target_proba_col[j]

        df = sg_df[sg_df[prior_col] == 0]

        y_true = df[target].values
        y_score = df[proba_col].values

        if len(np.unique(y_true)) < 2:
            continue

        # -----------------------------------
        # DeLong summary table
        # -----------------------------------
        auc, (ci_lower, ci_upper) = delong_auc_ci(y_true, y_score)

        delong_rows.append({
            'Subgroup': name,
            'Condition': target,
            'N': len(df),
            'AUC': round(auc, 6),
            'CI_Lower': round(ci_lower, 6),
            'CI_Upper': round(ci_upper, 6),
            'CI_Width': round(ci_upper - ci_lower, 6),
        })

        # -----------------------------------
        # Save raw predictions for ROC plots
        # -----------------------------------
        roc_rows.extend([
            {
                'Subgroup': name,
                'Target': target,
                'y_true': yt,
                'y_score': ys
            }
            for yt, ys in zip(y_true, y_score)
        ])

delong_df = pd.DataFrame(delong_rows)
roc_df = pd.DataFrame(roc_rows)

In [ ]:
saved_files = plot_roc_by_race(
    roc_df[roc_df['Subgroup'].isin(['Chinese', 'Malay', 'Indian', 'Others'])],
    disease_col="Target",
    race_col="Subgroup",
    truth_col="y_true",
    score_col="y_score"
)


## Other Metrics with 95% CI

In [ ]:
# =====================================================
# Convert summary table into long format
# =====================================================
# 1. Load Data
summary_df = pd.read_csv(f'{SUBDATAPATH}Performance_CI_all_subgroups_allinfo_nospace.csv')
summary_df = summary_df[summary_df['Subgroup'].isin(['Chinese', 'Malay', 'Indian', 'Others'])]
summary_df = summary_df[['Subgroup', 'Target', 'F1_bestperf', 'Precision_bestperf',
                         'Recall_bestperf', 'Specificity_bestperf', 'Accuracy_bestperf',
                         'F1_CI_lower', 'F1_CI_upper', 'Precision_CI_lower', 'Precision_CI_upper',
                         'Recall_CI_lower', 'Recall_CI_upper', 'Specificity_CI_lower',
                         'Specificity_CI_upper', 'Accuracy_CI_lower', 'Accuracy_CI_upper']].copy()
summary_df.columns = ['Subgroup', 'Target', 'F1', 'Precision', 'Recall', 'Specificity',
                      'Accuracy', 'F1_CI_lower', 'F1_CI_upper', 'Precision_CI_lower', 'Precision_CI_upper',
                      'Recall_CI_lower', 'Recall_CI_upper', 'Specificity_CI_lower',
                      'Specificity_CI_upper', 'Accuracy_CI_lower', 'Accuracy_CI_upper']

# 2. Format figdata
metrics = ["F1", "Precision", "Recall", "Specificity", "Accuracy"]
rows = []
for _, row in summary_df.iterrows():
    for metric in metrics:
        rows.append({
            "Subgroup": row["Subgroup"],
            "Target": row["Target"],
            "Metric": metric,
            "Mean": row[f"{metric}"],
            "Lower": row[f"{metric}_CI_lower"],
            "Upper": row[f"{metric}_CI_upper"]
        })
plot_df = pd.DataFrame(rows)


# =====================================================
# Plotting
# =====================================================
races = ["Chinese", "Malay", "Indian", "Others"]
colors = {"F1": "#E69F00", "Precision": "#009E73", "Recall": "#D55E00",
          "Specificity": "#CC79A7", "Accuracy": "#56B4E9"}
hatches = {"F1": "///", "Precision": "\\\\\\", "Recall": "xxx", "Specificity": "---",
           "Accuracy": "..."}
width = 0.12

for disease in sorted(plot_df["Target"].unique()):
    disease_df = plot_df[plot_df["Target"] == disease]

    fig, ax = plt.subplots(figsize=(10, 8))
    x = np.arange(len(races))
    for i, metric in enumerate(metrics):
        sub = disease_df[disease_df["Metric"] == metric].set_index("Subgroup")
        means = sub.loc[races, "Mean"]
        lower = sub.loc[races, "Lower"]
        upper = sub.loc[races, "Upper"]

        bars = ax.bar(x + i * width, means, width,
                      color=colors[metric], hatch=hatches[metric],
                      edgecolor='black', linewidth=0.8, label=metric)

        # Error bars
        ax.errorbar(x + i * width, means,
                    yerr=[means - lower, upper - means],
                    fmt='none', ecolor='black', capsize=3, linewidth=1)

        # Value labels above CI
        for bar, value, ci_upper in zip(bars, means, upper):
            ax.text(bar.get_x() + bar.get_width()/2, ci_upper + 0.02, f"{value:.3f}",
                    ha='center', va='bottom', rotation=90, fontsize=16)

    title_txt = disease
    if title_txt == 'Cerebral_Vascular_Accident':
        title_txt = 'Stroke'
    elif title_txt == 'Heart_and_Circulatory':
        title_txt = 'Heart Disease'
    else:
        title_txt = disease.replace("_", " ")
    ax.set_title(title_txt, fontsize=20, fontweight="bold")
    ax.set_ylabel("Performance", fontsize=20)
    ax.set_ylim(0, 1.15)
    ax.set_xticks(x + width * (len(metrics)-1)/2)
    ax.set_xticklabels(races, fontsize=20)
    ax.tick_params(axis="y", which="major", labelsize=20)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    # ax.legend(loc='lower right', fontsize=12, frameon=True)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12),
          ncol=5, fontsize=16, frameon=True)
    plt.tight_layout()

    # Save image
    filename = disease.replace("_", " ") + "_bars.png"

    plt.savefig(os.path.join(FIGPATH, filename), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()

print(f"Saved {len(plot_df['Target'].unique())} plots to {FIGPATH}")

# Table 3. New Data: Model Performance with 95% CIs

In [ ]:
%%time
NEWDATAFILE = 'DP_Phase4_combined_v3_newdata_wpreds.csv'
test_wpreds = pd.read_csv(PROCDATAPATH + NEWDATAFILE)
orgdata_2019 = pd.read_csv(DATAPATH2 + DATAFILE_2019, usecols=features_tbl)
orgdata_train = pd.read_csv(DATAPATH2 + DATAFILE_TRAIN, usecols=features_tbl)

# remove last 4 characters (i.e., 2019) from train_2019 pat_id
train_2019 = orgdata_2019.copy()
train_2019['PAT_ID'] = train_2019['PAT_ID'].str[:-4]
train_cohort = set(orgdata_train['PAT_ID'].unique()).union(set(train_2019['PAT_ID'].unique()))

## Prevalence Rate (New Patients)

In [ ]:
tmp = test_wpreds[~test_wpreds['PAT_ID'].isin(train_cohort)].copy()
for i in target_col[:-1]:
    print(f'{i}---------------')
    display(tmp[i+'_Prior'].value_counts(normalize=True))

## Specific-Defined Parameters

In [ ]:
N_BOOTSTRAPS    = 1000    # number of bootstrap iterations
SEED            = 42      # random seed for reproducibility
CI_LOWER        = 2.5     # lower percentile for 95% CI
CI_UPPER        = 97.5    # upper percentile for 95% CI
THRESHOLD_STEP  = 0.01    # step size for threshold search

# Sample size rule (per supervisor instruction):
#   subgroup size > 10,000 → sample 10,000 per iteration
#   subgroup size ≤ 10,000 → sample  5,000 per iteration
SAMPLE_SIZE_LARGE = 10000
SAMPLE_SIZE_SMALL =  5000
SAMPLE_SIZE_THRESHOLD = 10000

In [ ]:
target_order = [
    'Diabetes_Mellitus',
    'Dyslipidaemia',
    'Hypertension',
    'Cerebral_Vascular_Accident',
    'Heart_and_Circulatory',
    'Renal_Disease',
    'Osteoporosis',
    'Osteoarthritis'
]

In [ ]:
subgroups = [
    {
        'name':   'Overall',
        'filter': lambda df: df,
        'optimal_thresholds': dict(),
    }
]

## Best Performance

### Run for New Patients

In [ ]:
delong_rows = []
best_tbl = pd.DataFrame()

for sg in subgroups:
    name  = sg['name']
    sg_df = sg['filter'](test_wpreds[~test_wpreds['PAT_ID'].isin(train_cohort)]).reset_index(drop=True)
    print(f'---------{name}------------------')

    test_max_f1 = evaluate_model_from_scores(sg_df, sg_df[target_col].values,
                                             sg_df[target_proba_col].values, remove_prior=True)
    tbl = test_max_f1.iloc[[1,2,4,0,3,6,5,7]]
    tbl['Subgroup'] = name
    best_tbl = pd.concat([best_tbl, tbl], ignore_index=True)

best_tbl.to_csv(f'{SUBDATAPATH}/best_performance_newdata_newpats.csv', index=False)

## Bootstrap CI

### Run for New Patients

In [ ]:
all_ci = []  # collect all subgroup CI results for combined output

for sg in subgroups:
    name = sg['name']
    # if name in ['Overall', 'Chinese', 'Female', 'Male']:
    #     continue

    print(f'\n=== {name} ===')

    # Filter to subgroup
    sg_df = sg['filter'](test_wpreds[~test_wpreds['PAT_ID'].isin(train_cohort)]).reset_index(drop=True)
    n     = len(sg_df)
    if name in ['Chinese', 'Malay', 'Indian', 'Others']:  # Most ethnicities < 10K patients
        sample_size = SAMPLE_SIZE_SMALL
    else:
        sample_size = SAMPLE_SIZE_LARGE

    # Run bootstrap
    results, prior_removal_ratio = run_bootstrap(
        sg_df, target_col[:-1], target_prior_col, target_proba_col,
        sample_size
    )

    # Build CI dataframe
    ci_df = build_ci_df(results, target_col[:-1], name, n, sample_size)
    all_ci.append(ci_df)

    # Save per-subgroup results
    safe_name = name.replace(' ', '_').replace('/', '').replace('(', '').replace(')', '').replace('>', 'gte').replace('<', 'lt')
    ci_df.to_csv(f'{SUBDATAPATH}/bootstrap_CI_newdata_newpats_{safe_name}.csv', index=False)
    pd.DataFrame(prior_removal_ratio).to_csv(
        f'{SUBDATAPATH}/prior_removal_ratio_newdata_newpats_{safe_name}.csv', index=False
    )
    print(f'Saved: bootstrap_CI_newdata_newpats_{safe_name}.csv')

## Delong ROC-AUC CI

### Run for New Patients

In [ ]:
delong_rows = []

for sg in subgroups:
    name  = sg['name']
    sg_df = sg['filter'](test_wpreds[~test_wpreds['PAT_ID'].isin(train_cohort)]).reset_index(drop=True)

    for j, target in enumerate(target_col[:-1]):
        prior_col = target_prior_col[j]
        proba_col = target_proba_col[j]

        # Filter out prior patients (same as bootstrap)
        df = sg_df[sg_df[prior_col] == 0]

        y_true  = df[target].values
        y_score = df[proba_col].values

        if len(np.unique(y_true)) < 2:
            continue

        auc, (ci_lower, ci_upper) = delong_auc_ci(y_true, y_score)

        delong_rows.append({
            'Subgroup':   name,
            'Target':  target,
            'N':          len(df),
            'AUC':        round(auc,      6),
            'CI_Lower':   round(ci_lower, 6),
            'CI_Upper':   round(ci_upper, 6),
            'CI_Width':   round(ci_upper - ci_lower, 6),
        })

delong_df = pd.DataFrame(delong_rows)

# Set ordered categorical
delong_df['Target'] = pd.Categorical(
    delong_df['Target'],
    categories=target_order,
    ordered=True
)

# Sort by Subgroup first, then Condition order
delong_df = delong_df.sort_values(
    ['Subgroup', 'Target']
).reset_index(drop=True)

delong_df.to_csv(
    f'{SUBDATAPATH}/delong_AUC_CI_newdata_newpats.csv',
    index=False
)

## Combine

### Run for New Patients (Format)

In [ ]:
files = [i for i in os.listdir(SUBDATAPATH) if 'bootstrap_CI_newdata_newpats_' in i]
all_ci = []
for f in files:
    all_ci.append(pd.read_csv(SUBDATAPATH + f))
combined_ci = pd.concat(all_ci, ignore_index=True)

# Step 2. Set ordered categorical
combined_ci['Target'] = pd.Categorical(
    combined_ci['Target'],
    categories=target_order,
    ordered=True
)

# Step 3. Sort by Subgroup first, then Condition order
combined_ci = combined_ci.sort_values(
    ['Subgroup', 'Target']
).reset_index(drop=True)

# Step 4. Load Delong ROC-AUC CIs and best performance metrics
delong_ci = pd.read_csv(SUBDATAPATH + 'delong_AUC_CI_newdata_newpats.csv')
delong_ci.columns = [i + '_delong' if i in ['N', 'AUC', 'CI_Lower', 'CI_Upper', 'CI_Width'] else i for i in delong_ci.columns]
best_perf = pd.read_csv(SUBDATAPATH + 'best_performance_newdata_newpats.csv')
best_perf.rename(columns={'ROC-AUC':'AUC'}, inplace=True)
best_perf.columns = [i + '_bestperf' if i in ['AUC', 'Threshold', 'F1', 'Precision', 'Recall',
        'Specificity', 'Accuracy', 'Test_samples', 'Test_positive', 'Total_samples'] else i for i in best_perf.columns]

# Step 5: Merge additional columns from delong_ci and best_perf, matched on Subgroup + Target
combined_ci = combined_ci.merge(delong_ci, on=['Subgroup', 'Target'], how='left')
combined_ci = combined_ci.merge(best_perf, on=['Subgroup', 'Target'], how='left')

In [ ]:
all_formatted = []

for i, subgroup in enumerate(combined_ci['Subgroup'].drop_duplicates()):
    formatted = build_formatted_table(combined_ci, subgroup)
    formatted.insert(0, 'Subgroup', subgroup)  # keep track of which subgroup each block belongs to

    # Format target diseases
    formatted['Target Disease'] = ['Diabetes Mellitus', 'Dyslipidaemia', 'Hypertension', 'Stroke',
                                   'Heart Disease', 'Renal Disease', 'Osteoporosis', 'Osteoarthritis',
                                   'Simple Average']
    all_formatted.append(formatted)
    all_formatted.append(pd.DataFrame([{col: pd.NA for col in formatted.columns}] * 5))

# Combined CSV with all subgroups stacked, each block labeled by Subgroup
all_formatted_df = pd.concat(all_formatted, ignore_index=True)
all_formatted_df.to_excel(
    f'{PROCDATAPATH}/performance_CI_niceformat_newdata_newpats.xlsx', index=False)

# Fig 6. Feature Importance

## Specific Configuration

In [ ]:
os.environ["TF_USE_LEGACY_KERAS"] = "1"   # must be set before importing tensorflow
from allimport_xai_v3 import *
import pickle

In [ ]:
# input file
train_file = DATAPATH2 + 'DP_Phase4_combined_v3_train.csv'
train2019_file = DATAPATH2 + 'DP_Phase4_2019_ALL.csv'

TRAINPREDSFILE = 'DP_Phase4_combined_v3_train_wpreds.csv'
FEATUREMAPFILE = 'ACE_Feature_Map.csv'
TARGETMAPFILE = 'ACE_Target_Map.csv'
EXCLUSIONFILE = 'ACE_Final_Model_ExclusionFeatures.xlsx'

# Model files
model_name = '3m'
features_file = MODELPATH + 'saved_model/featurelist_{}.csv'.format(model_name)
scaler_file = MODELPATH + 'saved_model/scaler_{}.pkl'.format(model_name)
model_file = MODELPATH + 'saved_model/phase4_{}_model'.format(model_name)
kmeans_file = MODELPATH + 'saved_model/kmeans_{}.pkl'.format(model_name)
featdict_file = MODELPATH + 'saved_model/features_dict_{}.pkl'.format(model_name)

target_col = [
    'Cerebral_Vascular_Accident', 'Diabetes_Mellitus',
    'Dyslipidaemia', 'Heart_and_Circulatory', 'Hypertension',
    'Osteoporosis', 'Renal_Disease', 'Osteoarthritis', 'Deceased'
]

prior_diag_col = ["Cerebral_Vascular_Accident_Diagnosed","Cerebral_Vascular_Accident_Medication",
                  "Cerebral_Vascular_Accident_Test","Diabetes_Mellitus_Diagnosed","Diabetes_Mellitus_Medication",
                  "Diabetes_Mellitus_Test","Dyslipidaemia_Diagnosed","Dyslipidaemia_Medication","Dyslipidaemia_Test",
                  "Heart_and_Circulatory_Diagnosed","Heart_and_Circulatory_Medication","Heart_and_Circulatory_Test",
                  "Hypertension_Diagnosed","Hypertension_Medication","Hypertension_Test","Osteoporosis_Diagnosed",
                  "Osteoporosis_Medication","Osteoporosis_Test","Osteoarthritis_Diagnosed","Osteoarthritis_Medication",
                  "Osteoarthritis_Test","Renal_Disease_Diagnosed","Renal_Disease_Medication","Renal_Disease_Test",
                  "Deceased_Diagnosed","Deceased_Medication","Deceased_Test"
]

prior_diag_drop = ["Cerebral_Vascular_Accident_Test", "Cerebral_Vascular_Accident_Medication",
                  "Heart_and_Circulatory_Medication", "Heart_and_Circulatory_Test",
                  "Hypertension_Test",
                  "Osteoporosis_Test",
                  "Osteoarthritis_Medication", "Osteoarthritis_Test",
                  "Renal_Disease_Medication",
                  "Deceased_Diagnosed","Deceased_Medication","Deceased_Test"
]

target_prior_col = [target+'_Prior' for target in target_col[:-1]]

# SHAP parameters
non_periodic_features = ['PAT_AGE', 'GENDER_Male', 'GENDER_Female',
                         'RACE_Chinese', 'RACE_Malay', 'RACE_Indian', 'RACE_Others',
                         "Cancer_Prior","Cerebral_Vascular_Accident_Prior","Diabetes_Mellitus_Prior",
                         "Dyslipidaemia_Prior","Heart_and_Circulatory_Prior","Hypertension_Prior",
                         "Osteoporosis_Prior","Renal_Disease_Prior","OverWeight_Prior",
                         "Osteoarthritis_Prior","BPH_Prior","COPD_Prior","Epilepsy_Prior","HepB_Prior",
                         "Parkinson_Prior","Psoriasis_Prior","RA_Prior","Allergic_Prior","Asthma_Prior",
                         "Dementia_Prior","Gout_Prior"
] + prior_diag_col

# features to drop
feature_to_drop = ['PAT_ID'] + target_col + target_prior_col + prior_diag_drop

# validation data size
val_size=1000

BATCH_SIZE = 512
feature_periods = ['YEARH_', 'YEAR1_', 'YEAR1H_', 'YEAR2_', 'YEAR2H_', 'YEAR3_']

In [ ]:
count_cols = [
'YEARH_NUM_UNQ_ATCL2_A01', 'YEAR1_NUM_UNQ_ATCL2_A01', 'YEAR1H_NUM_UNQ_ATCL2_A01', 'YEAR2_NUM_UNQ_ATCL2_A01', 'YEAR2H_NUM_UNQ_ATCL2_A01', 'YEAR3_NUM_UNQ_ATCL2_A01',
'YEARH_NUM_UNQ_ATCL2_A02', 'YEAR1_NUM_UNQ_ATCL2_A02', 'YEAR1H_NUM_UNQ_ATCL2_A02', 'YEAR2_NUM_UNQ_ATCL2_A02', 'YEAR2H_NUM_UNQ_ATCL2_A02', 'YEAR3_NUM_UNQ_ATCL2_A02',
'YEARH_NUM_UNQ_ATCL2_A03', 'YEAR1_NUM_UNQ_ATCL2_A03', 'YEAR1H_NUM_UNQ_ATCL2_A03', 'YEAR2_NUM_UNQ_ATCL2_A03', 'YEAR2H_NUM_UNQ_ATCL2_A03', 'YEAR3_NUM_UNQ_ATCL2_A03',
'YEARH_NUM_UNQ_ATCL2_A04', 'YEAR1_NUM_UNQ_ATCL2_A04', 'YEAR1H_NUM_UNQ_ATCL2_A04', 'YEAR2_NUM_UNQ_ATCL2_A04', 'YEAR2H_NUM_UNQ_ATCL2_A04', 'YEAR3_NUM_UNQ_ATCL2_A04',
'YEARH_NUM_UNQ_ATCL2_A06', 'YEAR1_NUM_UNQ_ATCL2_A06', 'YEAR1H_NUM_UNQ_ATCL2_A06', 'YEAR2_NUM_UNQ_ATCL2_A06', 'YEAR2H_NUM_UNQ_ATCL2_A06', 'YEAR3_NUM_UNQ_ATCL2_A06',
'YEARH_NUM_UNQ_ATCL2_A07', 'YEAR1_NUM_UNQ_ATCL2_A07', 'YEAR1H_NUM_UNQ_ATCL2_A07', 'YEAR2_NUM_UNQ_ATCL2_A07', 'YEAR2H_NUM_UNQ_ATCL2_A07', 'YEAR3_NUM_UNQ_ATCL2_A07',
'YEARH_NUM_UNQ_ATCL2_A10', 'YEAR1_NUM_UNQ_ATCL2_A10', 'YEAR1H_NUM_UNQ_ATCL2_A10', 'YEAR2_NUM_UNQ_ATCL2_A10', 'YEAR2H_NUM_UNQ_ATCL2_A10', 'YEAR3_NUM_UNQ_ATCL2_A10',
'YEARH_NUM_UNQ_ATCL2_A11', 'YEAR1_NUM_UNQ_ATCL2_A11', 'YEAR1H_NUM_UNQ_ATCL2_A11', 'YEAR2_NUM_UNQ_ATCL2_A11', 'YEAR2H_NUM_UNQ_ATCL2_A11', 'YEAR3_NUM_UNQ_ATCL2_A11',
'YEARH_NUM_UNQ_ATCL2_A12', 'YEAR1_NUM_UNQ_ATCL2_A12', 'YEAR1H_NUM_UNQ_ATCL2_A12', 'YEAR2_NUM_UNQ_ATCL2_A12', 'YEAR2H_NUM_UNQ_ATCL2_A12', 'YEAR3_NUM_UNQ_ATCL2_A12',
'YEARH_NUM_UNQ_ATCL2_B01', 'YEAR1_NUM_UNQ_ATCL2_B01', 'YEAR1H_NUM_UNQ_ATCL2_B01', 'YEAR2_NUM_UNQ_ATCL2_B01', 'YEAR2H_NUM_UNQ_ATCL2_B01', 'YEAR3_NUM_UNQ_ATCL2_B01',
'YEARH_NUM_UNQ_ATCL2_B02', 'YEAR1_NUM_UNQ_ATCL2_B02', 'YEAR1H_NUM_UNQ_ATCL2_B02', 'YEAR2_NUM_UNQ_ATCL2_B02', 'YEAR2H_NUM_UNQ_ATCL2_B02', 'YEAR3_NUM_UNQ_ATCL2_B02',
'YEARH_NUM_UNQ_ATCL2_B03', 'YEAR1_NUM_UNQ_ATCL2_B03', 'YEAR1H_NUM_UNQ_ATCL2_B03', 'YEAR2_NUM_UNQ_ATCL2_B03', 'YEAR2H_NUM_UNQ_ATCL2_B03', 'YEAR3_NUM_UNQ_ATCL2_B03',
'YEARH_NUM_UNQ_ATCL2_B05', 'YEAR1_NUM_UNQ_ATCL2_B05', 'YEAR1H_NUM_UNQ_ATCL2_B05', 'YEAR2_NUM_UNQ_ATCL2_B05', 'YEAR2H_NUM_UNQ_ATCL2_B05', 'YEAR3_NUM_UNQ_ATCL2_B05',
'YEARH_NUM_UNQ_ATCL2_C01', 'YEAR1_NUM_UNQ_ATCL2_C01', 'YEAR1H_NUM_UNQ_ATCL2_C01', 'YEAR2_NUM_UNQ_ATCL2_C01', 'YEAR2H_NUM_UNQ_ATCL2_C01', 'YEAR3_NUM_UNQ_ATCL2_C01',
'YEARH_NUM_UNQ_ATCL2_C02', 'YEAR1_NUM_UNQ_ATCL2_C02', 'YEAR1H_NUM_UNQ_ATCL2_C02', 'YEAR2_NUM_UNQ_ATCL2_C02', 'YEAR2H_NUM_UNQ_ATCL2_C02', 'YEAR3_NUM_UNQ_ATCL2_C02',
'YEARH_NUM_UNQ_ATCL2_C03', 'YEAR1_NUM_UNQ_ATCL2_C03', 'YEAR1H_NUM_UNQ_ATCL2_C03', 'YEAR2_NUM_UNQ_ATCL2_C03', 'YEAR2H_NUM_UNQ_ATCL2_C03', 'YEAR3_NUM_UNQ_ATCL2_C03',
'YEARH_NUM_UNQ_ATCL2_C04', 'YEAR1_NUM_UNQ_ATCL2_C04', 'YEAR1H_NUM_UNQ_ATCL2_C04', 'YEAR2_NUM_UNQ_ATCL2_C04', 'YEAR2H_NUM_UNQ_ATCL2_C04', 'YEAR3_NUM_UNQ_ATCL2_C04',
'YEARH_NUM_UNQ_ATCL2_C05', 'YEAR1_NUM_UNQ_ATCL2_C05', 'YEAR1H_NUM_UNQ_ATCL2_C05', 'YEAR2_NUM_UNQ_ATCL2_C05', 'YEAR2H_NUM_UNQ_ATCL2_C05', 'YEAR3_NUM_UNQ_ATCL2_C05',
'YEARH_NUM_UNQ_ATCL2_C07', 'YEAR1_NUM_UNQ_ATCL2_C07', 'YEAR1H_NUM_UNQ_ATCL2_C07', 'YEAR2_NUM_UNQ_ATCL2_C07', 'YEAR2H_NUM_UNQ_ATCL2_C07', 'YEAR3_NUM_UNQ_ATCL2_C07',
'YEARH_NUM_UNQ_ATCL2_C08', 'YEAR1_NUM_UNQ_ATCL2_C08', 'YEAR1H_NUM_UNQ_ATCL2_C08', 'YEAR2_NUM_UNQ_ATCL2_C08', 'YEAR2H_NUM_UNQ_ATCL2_C08', 'YEAR3_NUM_UNQ_ATCL2_C08',
'YEARH_NUM_UNQ_ATCL2_C09', 'YEAR1_NUM_UNQ_ATCL2_C09', 'YEAR1H_NUM_UNQ_ATCL2_C09', 'YEAR2_NUM_UNQ_ATCL2_C09', 'YEAR2H_NUM_UNQ_ATCL2_C09', 'YEAR3_NUM_UNQ_ATCL2_C09',
'YEARH_NUM_UNQ_ATCL2_C10', 'YEAR1_NUM_UNQ_ATCL2_C10', 'YEAR1H_NUM_UNQ_ATCL2_C10', 'YEAR2_NUM_UNQ_ATCL2_C10', 'YEAR2H_NUM_UNQ_ATCL2_C10', 'YEAR3_NUM_UNQ_ATCL2_C10',
'YEARH_NUM_UNQ_ATCL2_D01', 'YEAR1_NUM_UNQ_ATCL2_D01', 'YEAR1H_NUM_UNQ_ATCL2_D01', 'YEAR2_NUM_UNQ_ATCL2_D01', 'YEAR2H_NUM_UNQ_ATCL2_D01', 'YEAR3_NUM_UNQ_ATCL2_D01',
'YEARH_NUM_UNQ_ATCL2_D02', 'YEAR1_NUM_UNQ_ATCL2_D02', 'YEAR1H_NUM_UNQ_ATCL2_D02', 'YEAR2_NUM_UNQ_ATCL2_D02', 'YEAR2H_NUM_UNQ_ATCL2_D02', 'YEAR3_NUM_UNQ_ATCL2_D02',
'YEARH_NUM_UNQ_ATCL2_D04', 'YEAR1_NUM_UNQ_ATCL2_D04', 'YEAR1H_NUM_UNQ_ATCL2_D04', 'YEAR2_NUM_UNQ_ATCL2_D04', 'YEAR2H_NUM_UNQ_ATCL2_D04', 'YEAR3_NUM_UNQ_ATCL2_D04',
'YEARH_NUM_UNQ_ATCL2_D06', 'YEAR1_NUM_UNQ_ATCL2_D06', 'YEAR1H_NUM_UNQ_ATCL2_D06', 'YEAR2_NUM_UNQ_ATCL2_D06', 'YEAR2H_NUM_UNQ_ATCL2_D06', 'YEAR3_NUM_UNQ_ATCL2_D06',
'YEARH_NUM_UNQ_ATCL2_D08', 'YEAR1_NUM_UNQ_ATCL2_D08', 'YEAR1H_NUM_UNQ_ATCL2_D08', 'YEAR2_NUM_UNQ_ATCL2_D08', 'YEAR2H_NUM_UNQ_ATCL2_D08', 'YEAR3_NUM_UNQ_ATCL2_D08',
'YEARH_NUM_UNQ_ATCL2_D11', 'YEAR1_NUM_UNQ_ATCL2_D11', 'YEAR1H_NUM_UNQ_ATCL2_D11', 'YEAR2_NUM_UNQ_ATCL2_D11', 'YEAR2H_NUM_UNQ_ATCL2_D11', 'YEAR3_NUM_UNQ_ATCL2_D11',
'YEARH_NUM_UNQ_ATCL2_G01', 'YEAR1_NUM_UNQ_ATCL2_G01', 'YEAR1H_NUM_UNQ_ATCL2_G01', 'YEAR2_NUM_UNQ_ATCL2_G01', 'YEAR2H_NUM_UNQ_ATCL2_G01', 'YEAR3_NUM_UNQ_ATCL2_G01',
'YEARH_NUM_UNQ_ATCL2_G03', 'YEAR1_NUM_UNQ_ATCL2_G03', 'YEAR1H_NUM_UNQ_ATCL2_G03', 'YEAR2_NUM_UNQ_ATCL2_G03', 'YEAR2H_NUM_UNQ_ATCL2_G03', 'YEAR3_NUM_UNQ_ATCL2_G03',
'YEARH_NUM_UNQ_ATCL2_G04', 'YEAR1_NUM_UNQ_ATCL2_G04', 'YEAR1H_NUM_UNQ_ATCL2_G04', 'YEAR2_NUM_UNQ_ATCL2_G04', 'YEAR2H_NUM_UNQ_ATCL2_G04', 'YEAR3_NUM_UNQ_ATCL2_G04',
'YEARH_NUM_UNQ_ATCL2_H01', 'YEAR1_NUM_UNQ_ATCL2_H01', 'YEAR1H_NUM_UNQ_ATCL2_H01', 'YEAR2_NUM_UNQ_ATCL2_H01', 'YEAR2H_NUM_UNQ_ATCL2_H01', 'YEAR3_NUM_UNQ_ATCL2_H01',
'YEARH_NUM_UNQ_ATCL2_H02', 'YEAR1_NUM_UNQ_ATCL2_H02', 'YEAR1H_NUM_UNQ_ATCL2_H02', 'YEAR2_NUM_UNQ_ATCL2_H02', 'YEAR2H_NUM_UNQ_ATCL2_H02', 'YEAR3_NUM_UNQ_ATCL2_H02',
'YEARH_NUM_UNQ_ATCL2_H03', 'YEAR1_NUM_UNQ_ATCL2_H03', 'YEAR1H_NUM_UNQ_ATCL2_H03', 'YEAR2_NUM_UNQ_ATCL2_H03', 'YEAR2H_NUM_UNQ_ATCL2_H03', 'YEAR3_NUM_UNQ_ATCL2_H03',
'YEARH_NUM_UNQ_ATCL2_H05', 'YEAR1_NUM_UNQ_ATCL2_H05', 'YEAR1H_NUM_UNQ_ATCL2_H05', 'YEAR2_NUM_UNQ_ATCL2_H05', 'YEAR2H_NUM_UNQ_ATCL2_H05', 'YEAR3_NUM_UNQ_ATCL2_H05',
'YEARH_NUM_UNQ_ATCL2_L01', 'YEAR1_NUM_UNQ_ATCL2_L01', 'YEAR1H_NUM_UNQ_ATCL2_L01', 'YEAR2_NUM_UNQ_ATCL2_L01', 'YEAR2H_NUM_UNQ_ATCL2_L01', 'YEAR3_NUM_UNQ_ATCL2_L01',
'YEARH_NUM_UNQ_ATCL2_L02', 'YEAR1_NUM_UNQ_ATCL2_L02', 'YEAR1H_NUM_UNQ_ATCL2_L02', 'YEAR2_NUM_UNQ_ATCL2_L02', 'YEAR2H_NUM_UNQ_ATCL2_L02', 'YEAR3_NUM_UNQ_ATCL2_L02',
'YEARH_NUM_UNQ_ATCL2_L04', 'YEAR1_NUM_UNQ_ATCL2_L04', 'YEAR1H_NUM_UNQ_ATCL2_L04', 'YEAR2_NUM_UNQ_ATCL2_L04', 'YEAR2H_NUM_UNQ_ATCL2_L04', 'YEAR3_NUM_UNQ_ATCL2_L04',
'YEARH_NUM_UNQ_ATCL2_M01', 'YEAR1_NUM_UNQ_ATCL2_M01', 'YEAR1H_NUM_UNQ_ATCL2_M01', 'YEAR2_NUM_UNQ_ATCL2_M01', 'YEAR2H_NUM_UNQ_ATCL2_M01', 'YEAR3_NUM_UNQ_ATCL2_M01',
'YEARH_NUM_UNQ_ATCL2_M02', 'YEAR1_NUM_UNQ_ATCL2_M02', 'YEAR1H_NUM_UNQ_ATCL2_M02', 'YEAR2_NUM_UNQ_ATCL2_M02', 'YEAR2H_NUM_UNQ_ATCL2_M02', 'YEAR3_NUM_UNQ_ATCL2_M02',
'YEARH_NUM_UNQ_ATCL2_M03', 'YEAR1_NUM_UNQ_ATCL2_M03', 'YEAR1H_NUM_UNQ_ATCL2_M03', 'YEAR2_NUM_UNQ_ATCL2_M03', 'YEAR2H_NUM_UNQ_ATCL2_M03', 'YEAR3_NUM_UNQ_ATCL2_M03',
'YEARH_NUM_UNQ_ATCL2_M04', 'YEAR1_NUM_UNQ_ATCL2_M04', 'YEAR1H_NUM_UNQ_ATCL2_M04', 'YEAR2_NUM_UNQ_ATCL2_M04', 'YEAR2H_NUM_UNQ_ATCL2_M04', 'YEAR3_NUM_UNQ_ATCL2_M04',
'YEARH_NUM_UNQ_ATCL2_M05', 'YEAR1_NUM_UNQ_ATCL2_M05', 'YEAR1H_NUM_UNQ_ATCL2_M05', 'YEAR2_NUM_UNQ_ATCL2_M05', 'YEAR2H_NUM_UNQ_ATCL2_M05', 'YEAR3_NUM_UNQ_ATCL2_M05',
'YEARH_NUM_UNQ_ATCL2_N02', 'YEAR1_NUM_UNQ_ATCL2_N02', 'YEAR1H_NUM_UNQ_ATCL2_N02', 'YEAR2_NUM_UNQ_ATCL2_N02', 'YEAR2H_NUM_UNQ_ATCL2_N02', 'YEAR3_NUM_UNQ_ATCL2_N02',
'YEARH_NUM_UNQ_ATCL2_N03', 'YEAR1_NUM_UNQ_ATCL2_N03', 'YEAR1H_NUM_UNQ_ATCL2_N03', 'YEAR2_NUM_UNQ_ATCL2_N03', 'YEAR2H_NUM_UNQ_ATCL2_N03', 'YEAR3_NUM_UNQ_ATCL2_N03',
'YEARH_NUM_UNQ_ATCL2_N04', 'YEAR1_NUM_UNQ_ATCL2_N04', 'YEAR1H_NUM_UNQ_ATCL2_N04', 'YEAR2_NUM_UNQ_ATCL2_N04', 'YEAR2H_NUM_UNQ_ATCL2_N04', 'YEAR3_NUM_UNQ_ATCL2_N04',
'YEARH_NUM_UNQ_ATCL2_N05', 'YEAR1_NUM_UNQ_ATCL2_N05', 'YEAR1H_NUM_UNQ_ATCL2_N05', 'YEAR2_NUM_UNQ_ATCL2_N05', 'YEAR2H_NUM_UNQ_ATCL2_N05', 'YEAR3_NUM_UNQ_ATCL2_N05',
'YEARH_NUM_UNQ_ATCL2_N06', 'YEAR1_NUM_UNQ_ATCL2_N06', 'YEAR1H_NUM_UNQ_ATCL2_N06', 'YEAR2_NUM_UNQ_ATCL2_N06', 'YEAR2H_NUM_UNQ_ATCL2_N06', 'YEAR3_NUM_UNQ_ATCL2_N06',
'YEARH_NUM_UNQ_ATCL2_R01', 'YEAR1_NUM_UNQ_ATCL2_R01', 'YEAR1H_NUM_UNQ_ATCL2_R01', 'YEAR2_NUM_UNQ_ATCL2_R01', 'YEAR2H_NUM_UNQ_ATCL2_R01', 'YEAR3_NUM_UNQ_ATCL2_R01',
'YEARH_NUM_UNQ_ATCL2_R02', 'YEAR1_NUM_UNQ_ATCL2_R02', 'YEAR1H_NUM_UNQ_ATCL2_R02', 'YEAR2_NUM_UNQ_ATCL2_R02', 'YEAR2H_NUM_UNQ_ATCL2_R02', 'YEAR3_NUM_UNQ_ATCL2_R02',
'YEARH_NUM_UNQ_ATCL2_R03', 'YEAR1_NUM_UNQ_ATCL2_R03', 'YEAR1H_NUM_UNQ_ATCL2_R03', 'YEAR2_NUM_UNQ_ATCL2_R03', 'YEAR2H_NUM_UNQ_ATCL2_R03', 'YEAR3_NUM_UNQ_ATCL2_R03',
'YEARH_NUM_UNQ_ATCL2_R05', 'YEAR1_NUM_UNQ_ATCL2_R05', 'YEAR1H_NUM_UNQ_ATCL2_R05', 'YEAR2_NUM_UNQ_ATCL2_R05', 'YEAR2H_NUM_UNQ_ATCL2_R05', 'YEAR3_NUM_UNQ_ATCL2_R05',
'YEARH_NUM_UNQ_ATCL2_R06', 'YEAR1_NUM_UNQ_ATCL2_R06', 'YEAR1H_NUM_UNQ_ATCL2_R06', 'YEAR2_NUM_UNQ_ATCL2_R06', 'YEAR2H_NUM_UNQ_ATCL2_R06', 'YEAR3_NUM_UNQ_ATCL2_R06',
'YEARH_NUM_UNQ_ATCL2_S01', 'YEAR1_NUM_UNQ_ATCL2_S01', 'YEAR1H_NUM_UNQ_ATCL2_S01', 'YEAR2_NUM_UNQ_ATCL2_S01', 'YEAR2H_NUM_UNQ_ATCL2_S01', 'YEAR3_NUM_UNQ_ATCL2_S01',
'YEARH_NUM_UNQ_ATCL2_S03', 'YEAR1_NUM_UNQ_ATCL2_S03', 'YEAR1H_NUM_UNQ_ATCL2_S03', 'YEAR2_NUM_UNQ_ATCL2_S03', 'YEAR2H_NUM_UNQ_ATCL2_S03', 'YEAR3_NUM_UNQ_ATCL2_S03',
'YEARH_UNQ_SUBCHAPTER_blood1', 'YEAR1_UNQ_SUBCHAPTER_blood1', 'YEAR1H_UNQ_SUBCHAPTER_blood1', 'YEAR2_UNQ_SUBCHAPTER_blood1', 'YEAR2H_UNQ_SUBCHAPTER_blood1', 'YEAR3_UNQ_SUBCHAPTER_blood1',
'YEARH_UNQ_SUBCHAPTER_blood2', 'YEAR1_UNQ_SUBCHAPTER_blood2', 'YEAR1H_UNQ_SUBCHAPTER_blood2', 'YEAR2_UNQ_SUBCHAPTER_blood2', 'YEAR2H_UNQ_SUBCHAPTER_blood2', 'YEAR3_UNQ_SUBCHAPTER_blood2',
'YEARH_UNQ_SUBCHAPTER_circulatory3', 'YEAR1_UNQ_SUBCHAPTER_circulatory3', 'YEAR1H_UNQ_SUBCHAPTER_circulatory3', 'YEAR2_UNQ_SUBCHAPTER_circulatory3', 'YEAR2H_UNQ_SUBCHAPTER_circulatory3', 'YEAR3_UNQ_SUBCHAPTER_circulatory3',
'YEARH_UNQ_SUBCHAPTER_circulatory4', 'YEAR1_UNQ_SUBCHAPTER_circulatory4', 'YEAR1H_UNQ_SUBCHAPTER_circulatory4', 'YEAR2_UNQ_SUBCHAPTER_circulatory4', 'YEAR2H_UNQ_SUBCHAPTER_circulatory4', 'YEAR3_UNQ_SUBCHAPTER_circulatory4',
'YEARH_UNQ_SUBCHAPTER_circulatory5', 'YEAR1_UNQ_SUBCHAPTER_circulatory5', 'YEAR1H_UNQ_SUBCHAPTER_circulatory5', 'YEAR2_UNQ_SUBCHAPTER_circulatory5', 'YEAR2H_UNQ_SUBCHAPTER_circulatory5', 'YEAR3_UNQ_SUBCHAPTER_circulatory5',
'YEARH_UNQ_SUBCHAPTER_circulatory6', 'YEAR1_UNQ_SUBCHAPTER_circulatory6', 'YEAR1H_UNQ_SUBCHAPTER_circulatory6', 'YEAR2_UNQ_SUBCHAPTER_circulatory6', 'YEAR2H_UNQ_SUBCHAPTER_circulatory6', 'YEAR3_UNQ_SUBCHAPTER_circulatory6',
'YEARH_UNQ_SUBCHAPTER_circulatory7', 'YEAR1_UNQ_SUBCHAPTER_circulatory7', 'YEAR1H_UNQ_SUBCHAPTER_circulatory7', 'YEAR2_UNQ_SUBCHAPTER_circulatory7', 'YEAR2H_UNQ_SUBCHAPTER_circulatory7', 'YEAR3_UNQ_SUBCHAPTER_circulatory7',
'YEARH_UNQ_SUBCHAPTER_circulatory8', 'YEAR1_UNQ_SUBCHAPTER_circulatory8', 'YEAR1H_UNQ_SUBCHAPTER_circulatory8', 'YEAR2_UNQ_SUBCHAPTER_circulatory8', 'YEAR2H_UNQ_SUBCHAPTER_circulatory8', 'YEAR3_UNQ_SUBCHAPTER_circulatory8',
'YEARH_UNQ_SUBCHAPTER_circulatory9', 'YEAR1_UNQ_SUBCHAPTER_circulatory9', 'YEAR1H_UNQ_SUBCHAPTER_circulatory9', 'YEAR2_UNQ_SUBCHAPTER_circulatory9', 'YEAR2H_UNQ_SUBCHAPTER_circulatory9', 'YEAR3_UNQ_SUBCHAPTER_circulatory9',
'YEARH_UNQ_SUBCHAPTER_digestive2', 'YEAR1_UNQ_SUBCHAPTER_digestive2', 'YEAR1H_UNQ_SUBCHAPTER_digestive2', 'YEAR2_UNQ_SUBCHAPTER_digestive2', 'YEAR2H_UNQ_SUBCHAPTER_digestive2', 'YEAR3_UNQ_SUBCHAPTER_digestive2',
'YEARH_UNQ_SUBCHAPTER_digestive5', 'YEAR1_UNQ_SUBCHAPTER_digestive5', 'YEAR1H_UNQ_SUBCHAPTER_digestive5', 'YEAR2_UNQ_SUBCHAPTER_digestive5', 'YEAR2H_UNQ_SUBCHAPTER_digestive5', 'YEAR3_UNQ_SUBCHAPTER_digestive5',
'YEARH_UNQ_SUBCHAPTER_digestive6', 'YEAR1_UNQ_SUBCHAPTER_digestive6', 'YEAR1H_UNQ_SUBCHAPTER_digestive6', 'YEAR2_UNQ_SUBCHAPTER_digestive6', 'YEAR2H_UNQ_SUBCHAPTER_digestive6', 'YEAR3_UNQ_SUBCHAPTER_digestive6',
'YEARH_UNQ_SUBCHAPTER_digestive7', 'YEAR1_UNQ_SUBCHAPTER_digestive7', 'YEAR1H_UNQ_SUBCHAPTER_digestive7', 'YEAR2_UNQ_SUBCHAPTER_digestive7', 'YEAR2H_UNQ_SUBCHAPTER_digestive7', 'YEAR3_UNQ_SUBCHAPTER_digestive7',
'YEARH_UNQ_SUBCHAPTER_genitourinary1', 'YEAR1_UNQ_SUBCHAPTER_genitourinary1', 'YEAR1H_UNQ_SUBCHAPTER_genitourinary1', 'YEAR2_UNQ_SUBCHAPTER_genitourinary1', 'YEAR2H_UNQ_SUBCHAPTER_genitourinary1', 'YEAR3_UNQ_SUBCHAPTER_genitourinary1',
'YEARH_UNQ_SUBCHAPTER_genitourinary2', 'YEAR1_UNQ_SUBCHAPTER_genitourinary2', 'YEAR1H_UNQ_SUBCHAPTER_genitourinary2', 'YEAR2_UNQ_SUBCHAPTER_genitourinary2', 'YEAR2H_UNQ_SUBCHAPTER_genitourinary2', 'YEAR3_UNQ_SUBCHAPTER_genitourinary2',
'YEARH_UNQ_SUBCHAPTER_genitourinary3', 'YEAR1_UNQ_SUBCHAPTER_genitourinary3', 'YEAR1H_UNQ_SUBCHAPTER_genitourinary3', 'YEAR2_UNQ_SUBCHAPTER_genitourinary3', 'YEAR2H_UNQ_SUBCHAPTER_genitourinary3', 'YEAR3_UNQ_SUBCHAPTER_genitourinary3',
'YEARH_UNQ_SUBCHAPTER_genitourinary4', 'YEAR1_UNQ_SUBCHAPTER_genitourinary4', 'YEAR1H_UNQ_SUBCHAPTER_genitourinary4', 'YEAR2_UNQ_SUBCHAPTER_genitourinary4', 'YEAR2H_UNQ_SUBCHAPTER_genitourinary4', 'YEAR3_UNQ_SUBCHAPTER_genitourinary4',
'YEARH_UNQ_SUBCHAPTER_genitourinary6', 'YEAR1_UNQ_SUBCHAPTER_genitourinary6', 'YEAR1H_UNQ_SUBCHAPTER_genitourinary6', 'YEAR2_UNQ_SUBCHAPTER_genitourinary6', 'YEAR2H_UNQ_SUBCHAPTER_genitourinary6', 'YEAR3_UNQ_SUBCHAPTER_genitourinary6',
'YEARH_UNQ_SUBCHAPTER_metabolic1', 'YEAR1_UNQ_SUBCHAPTER_metabolic1', 'YEAR1H_UNQ_SUBCHAPTER_metabolic1', 'YEAR2_UNQ_SUBCHAPTER_metabolic1', 'YEAR2H_UNQ_SUBCHAPTER_metabolic1', 'YEAR3_UNQ_SUBCHAPTER_metabolic1',
'YEARH_UNQ_SUBCHAPTER_metabolic2', 'YEAR1_UNQ_SUBCHAPTER_metabolic2', 'YEAR1H_UNQ_SUBCHAPTER_metabolic2', 'YEAR2_UNQ_SUBCHAPTER_metabolic2', 'YEAR2H_UNQ_SUBCHAPTER_metabolic2', 'YEAR3_UNQ_SUBCHAPTER_metabolic2',
'YEARH_UNQ_SUBCHAPTER_metabolic3', 'YEAR1_UNQ_SUBCHAPTER_metabolic3', 'YEAR1H_UNQ_SUBCHAPTER_metabolic3', 'YEAR2_UNQ_SUBCHAPTER_metabolic3', 'YEAR2H_UNQ_SUBCHAPTER_metabolic3', 'YEAR3_UNQ_SUBCHAPTER_metabolic3',
'YEARH_UNQ_SUBCHAPTER_metabolic4', 'YEAR1_UNQ_SUBCHAPTER_metabolic4', 'YEAR1H_UNQ_SUBCHAPTER_metabolic4', 'YEAR2_UNQ_SUBCHAPTER_metabolic4', 'YEAR2H_UNQ_SUBCHAPTER_metabolic4', 'YEAR3_UNQ_SUBCHAPTER_metabolic4',
'YEARH_UNQ_SUBCHAPTER_musculoskeletal1', 'YEAR1_UNQ_SUBCHAPTER_musculoskeletal1', 'YEAR1H_UNQ_SUBCHAPTER_musculoskeletal1', 'YEAR2_UNQ_SUBCHAPTER_musculoskeletal1', 'YEAR2H_UNQ_SUBCHAPTER_musculoskeletal1', 'YEAR3_UNQ_SUBCHAPTER_musculoskeletal1',
'YEARH_UNQ_SUBCHAPTER_musculoskeletal2', 'YEAR1_UNQ_SUBCHAPTER_musculoskeletal2', 'YEAR1H_UNQ_SUBCHAPTER_musculoskeletal2', 'YEAR2_UNQ_SUBCHAPTER_musculoskeletal2', 'YEAR2H_UNQ_SUBCHAPTER_musculoskeletal2', 'YEAR3_UNQ_SUBCHAPTER_musculoskeletal2',
'YEARH_UNQ_SUBCHAPTER_musculoskeletal3', 'YEAR1_UNQ_SUBCHAPTER_musculoskeletal3', 'YEAR1H_UNQ_SUBCHAPTER_musculoskeletal3', 'YEAR2_UNQ_SUBCHAPTER_musculoskeletal3', 'YEAR2H_UNQ_SUBCHAPTER_musculoskeletal3', 'YEAR3_UNQ_SUBCHAPTER_musculoskeletal3',
'YEARH_UNQ_SUBCHAPTER_musculoskeletal4', 'YEAR1_UNQ_SUBCHAPTER_musculoskeletal4', 'YEAR1H_UNQ_SUBCHAPTER_musculoskeletal4', 'YEAR2_UNQ_SUBCHAPTER_musculoskeletal4', 'YEAR2H_UNQ_SUBCHAPTER_musculoskeletal4', 'YEAR3_UNQ_SUBCHAPTER_musculoskeletal4',
'YEARH_UNQ_SUBCHAPTER_neoplasm12', 'YEAR1_UNQ_SUBCHAPTER_neoplasm12', 'YEAR1H_UNQ_SUBCHAPTER_neoplasm12', 'YEAR2_UNQ_SUBCHAPTER_neoplasm12', 'YEAR2H_UNQ_SUBCHAPTER_neoplasm12', 'YEAR3_UNQ_SUBCHAPTER_neoplasm12',
'YEARH_UNQ_SUBCHAPTER_neoplasm2', 'YEAR1_UNQ_SUBCHAPTER_neoplasm2', 'YEAR1H_UNQ_SUBCHAPTER_neoplasm2', 'YEAR2_UNQ_SUBCHAPTER_neoplasm2', 'YEAR2H_UNQ_SUBCHAPTER_neoplasm2', 'YEAR3_UNQ_SUBCHAPTER_neoplasm2',
'YEARH_UNQ_SUBCHAPTER_neoplasm3', 'YEAR1_UNQ_SUBCHAPTER_neoplasm3', 'YEAR1H_UNQ_SUBCHAPTER_neoplasm3', 'YEAR2_UNQ_SUBCHAPTER_neoplasm3', 'YEAR2H_UNQ_SUBCHAPTER_neoplasm3', 'YEAR3_UNQ_SUBCHAPTER_neoplasm3',
'YEARH_UNQ_SUBCHAPTER_neoplasm4', 'YEAR1_UNQ_SUBCHAPTER_neoplasm4', 'YEAR1H_UNQ_SUBCHAPTER_neoplasm4', 'YEAR2_UNQ_SUBCHAPTER_neoplasm4', 'YEAR2H_UNQ_SUBCHAPTER_neoplasm4', 'YEAR3_UNQ_SUBCHAPTER_neoplasm4',
'YEARH_UNQ_SUBCHAPTER_neoplasm6', 'YEAR1_UNQ_SUBCHAPTER_neoplasm6', 'YEAR1H_UNQ_SUBCHAPTER_neoplasm6', 'YEAR2_UNQ_SUBCHAPTER_neoplasm6', 'YEAR2H_UNQ_SUBCHAPTER_neoplasm6', 'YEAR3_UNQ_SUBCHAPTER_neoplasm6',
'YEARH_UNQ_SUBCHAPTER_neoplasm7', 'YEAR1_UNQ_SUBCHAPTER_neoplasm7', 'YEAR1H_UNQ_SUBCHAPTER_neoplasm7', 'YEAR2_UNQ_SUBCHAPTER_neoplasm7', 'YEAR2H_UNQ_SUBCHAPTER_neoplasm7', 'YEAR3_UNQ_SUBCHAPTER_neoplasm7',
'YEARH_UNQ_SUBCHAPTER_neoplasm9', 'YEAR1_UNQ_SUBCHAPTER_neoplasm9', 'YEAR1H_UNQ_SUBCHAPTER_neoplasm9', 'YEAR2_UNQ_SUBCHAPTER_neoplasm9', 'YEAR2H_UNQ_SUBCHAPTER_neoplasm9', 'YEAR3_UNQ_SUBCHAPTER_neoplasm9',
'YEARH_UNQ_SUBCHAPTER_nervous2', 'YEAR1_UNQ_SUBCHAPTER_nervous2', 'YEAR1H_UNQ_SUBCHAPTER_nervous2', 'YEAR2_UNQ_SUBCHAPTER_nervous2', 'YEAR2H_UNQ_SUBCHAPTER_nervous2', 'YEAR3_UNQ_SUBCHAPTER_nervous2',
'YEARH_UNQ_SUBCHAPTER_nervous3', 'YEAR1_UNQ_SUBCHAPTER_nervous3', 'YEAR1H_UNQ_SUBCHAPTER_nervous3', 'YEAR2_UNQ_SUBCHAPTER_nervous3', 'YEAR2H_UNQ_SUBCHAPTER_nervous3', 'YEAR3_UNQ_SUBCHAPTER_nervous3',
'YEARH_UNQ_SUBCHAPTER_nervous4', 'YEAR1_UNQ_SUBCHAPTER_nervous4', 'YEAR1H_UNQ_SUBCHAPTER_nervous4', 'YEAR2_UNQ_SUBCHAPTER_nervous4', 'YEAR2H_UNQ_SUBCHAPTER_nervous4', 'YEAR3_UNQ_SUBCHAPTER_nervous4',
'YEARH_UNQ_SUBCHAPTER_nervous5', 'YEAR1_UNQ_SUBCHAPTER_nervous5', 'YEAR1H_UNQ_SUBCHAPTER_nervous5', 'YEAR2_UNQ_SUBCHAPTER_nervous5', 'YEAR2H_UNQ_SUBCHAPTER_nervous5', 'YEAR3_UNQ_SUBCHAPTER_nervous5',
'YEARH_UNQ_SUBCHAPTER_nervous7', 'YEAR1_UNQ_SUBCHAPTER_nervous7', 'YEAR1H_UNQ_SUBCHAPTER_nervous7', 'YEAR2_UNQ_SUBCHAPTER_nervous7', 'YEAR2H_UNQ_SUBCHAPTER_nervous7', 'YEAR3_UNQ_SUBCHAPTER_nervous7',
'YEARH_UNQ_SUBCHAPTER_nervous8', 'YEAR1_UNQ_SUBCHAPTER_nervous8', 'YEAR1H_UNQ_SUBCHAPTER_nervous8', 'YEAR2_UNQ_SUBCHAPTER_nervous8', 'YEAR2H_UNQ_SUBCHAPTER_nervous8', 'YEAR3_UNQ_SUBCHAPTER_nervous8',
'YEARH_UNQ_SUBCHAPTER_respiratory1', 'YEAR1_UNQ_SUBCHAPTER_respiratory1', 'YEAR1H_UNQ_SUBCHAPTER_respiratory1', 'YEAR2_UNQ_SUBCHAPTER_respiratory1', 'YEAR2H_UNQ_SUBCHAPTER_respiratory1', 'YEAR3_UNQ_SUBCHAPTER_respiratory1',
'YEARH_UNQ_SUBCHAPTER_respiratory3', 'YEAR1_UNQ_SUBCHAPTER_respiratory3', 'YEAR1H_UNQ_SUBCHAPTER_respiratory3', 'YEAR2_UNQ_SUBCHAPTER_respiratory3', 'YEAR2H_UNQ_SUBCHAPTER_respiratory3', 'YEAR3_UNQ_SUBCHAPTER_respiratory3',
'YEARH_UNQ_SUBCHAPTER_respiratory4', 'YEAR1_UNQ_SUBCHAPTER_respiratory4', 'YEAR1H_UNQ_SUBCHAPTER_respiratory4', 'YEAR2_UNQ_SUBCHAPTER_respiratory4', 'YEAR2H_UNQ_SUBCHAPTER_respiratory4', 'YEAR3_UNQ_SUBCHAPTER_respiratory4',
'YEARH_UNQ_SUBCHAPTER_respiratory5', 'YEAR1_UNQ_SUBCHAPTER_respiratory5', 'YEAR1H_UNQ_SUBCHAPTER_respiratory5', 'YEAR2_UNQ_SUBCHAPTER_respiratory5', 'YEAR2H_UNQ_SUBCHAPTER_respiratory5', 'YEAR3_UNQ_SUBCHAPTER_respiratory5',
'YEARH_UNQ_SUBCHAPTER_respiratory6', 'YEAR1_UNQ_SUBCHAPTER_respiratory6', 'YEAR1H_UNQ_SUBCHAPTER_respiratory6', 'YEAR2_UNQ_SUBCHAPTER_respiratory6', 'YEAR2H_UNQ_SUBCHAPTER_respiratory6', 'YEAR3_UNQ_SUBCHAPTER_respiratory6',
'YEARH_UNQ_SUBCHAPTER_skin1', 'YEAR1_UNQ_SUBCHAPTER_skin1', 'YEAR1H_UNQ_SUBCHAPTER_skin1', 'YEAR2_UNQ_SUBCHAPTER_skin1', 'YEAR2H_UNQ_SUBCHAPTER_skin1', 'YEAR3_UNQ_SUBCHAPTER_skin1',
'YEARH_UNQ_SUBCHAPTER_skin2', 'YEAR1_UNQ_SUBCHAPTER_skin2', 'YEAR1H_UNQ_SUBCHAPTER_skin2', 'YEAR2_UNQ_SUBCHAPTER_skin2', 'YEAR2H_UNQ_SUBCHAPTER_skin2', 'YEAR3_UNQ_SUBCHAPTER_skin2',
'YEARH_UNQ_SUBCHAPTER_skin3', 'YEAR1_UNQ_SUBCHAPTER_skin3', 'YEAR1H_UNQ_SUBCHAPTER_skin3', 'YEAR2_UNQ_SUBCHAPTER_skin3', 'YEAR2H_UNQ_SUBCHAPTER_skin3', 'YEAR3_UNQ_SUBCHAPTER_skin3',
'YEARH_INP_COUNT', 'YEARH_INP_TLOS', 'YEARH_ED_COUNT', 'YEARH_SOC_COUNT',
'YEARH_ORTHOPAEDIC_SURGERY_COUNT', 'YEARH_CARDIOLOGY_COUNT', 'YEARH_FAMILY_GENERAL_COUNT',
'YEARH_OPHTHALMOLOGY_COUNT', 'YEARH_MEDICAL_ONCOLOGY_COUNT', 'YEARH_UROLOGY_COUNT',
'YEAR1_INP_COUNT', 'YEAR1_INP_TLOS', 'YEAR1_ED_COUNT', 'YEAR1_SOC_COUNT',
'YEAR1_ORTHOPAEDIC_SURGERY_COUNT', 'YEAR1_CARDIOLOGY_COUNT', 'YEAR1_FAMILY_GENERAL_COUNT',
'YEAR1_OPHTHALMOLOGY_COUNT', 'YEAR1_MEDICAL_ONCOLOGY_COUNT', 'YEAR1_UROLOGY_COUNT',
'YEAR1H_INP_COUNT', 'YEAR1H_INP_TLOS', 'YEAR1H_ED_COUNT', 'YEAR1H_SOC_COUNT',
'YEAR1H_ORTHOPAEDIC_SURGERY_COUNT', 'YEAR1H_CARDIOLOGY_COUNT', 'YEAR1H_FAMILY_GENERAL_COUNT',
'YEAR1H_OPHTHALMOLOGY_COUNT', 'YEAR1H_MEDICAL_ONCOLOGY_COUNT', 'YEAR1H_UROLOGY_COUNT',
'YEAR2_INP_COUNT', 'YEAR2_INP_TLOS', 'YEAR2_ED_COUNT', 'YEAR2_SOC_COUNT',
'YEAR2_ORTHOPAEDIC_SURGERY_COUNT', 'YEAR2_CARDIOLOGY_COUNT', 'YEAR2_FAMILY_GENERAL_COUNT',
'YEAR2_OPHTHALMOLOGY_COUNT', 'YEAR2_MEDICAL_ONCOLOGY_COUNT', 'YEAR2_UROLOGY_COUNT',
'YEAR2H_INP_COUNT', 'YEAR2H_INP_TLOS', 'YEAR2H_ED_COUNT', 'YEAR2H_SOC_COUNT',
'YEAR2H_ORTHOPAEDIC_SURGERY_COUNT', 'YEAR2H_CARDIOLOGY_COUNT', 'YEAR2H_FAMILY_GENERAL_COUNT',
'YEAR2H_OPHTHALMOLOGY_COUNT', 'YEAR2H_MEDICAL_ONCOLOGY_COUNT', 'YEAR2H_UROLOGY_COUNT',
'YEAR3_INP_COUNT', 'YEAR3_INP_TLOS', 'YEAR3_ED_COUNT', 'YEAR3_SOC_COUNT',
'YEAR3_ORTHOPAEDIC_SURGERY_COUNT', 'YEAR3_CARDIOLOGY_COUNT', 'YEAR3_FAMILY_GENERAL_COUNT',
'YEAR3_OPHTHALMOLOGY_COUNT', 'YEAR3_MEDICAL_ONCOLOGY_COUNT', 'YEAR3_UROLOGY_COUNT'
]

## Update on DeepPatient Due to Latest Version of SHAP and plots

In [ ]:
# Imports & Declarations
import numpy as np
import pandas as pd
import shap
from tensorflow import keras
from pickle import load
import matplotlib.pyplot as plt

random_state = 1234
np.random.seed(random_state)


class DeepPatientShap:
    """
    Methods
    -------
    shap_explainer_generator: fit a KernelExplainer for one row
    shap_top_features: signed SHAP table for one patient
    plot_shap_force: force plot for one patient
    shap_explainer_generator_multiple: fit a KernelExplainer over many rows
    shap_top_features_multiple: mean |SHAP| ranking for a target
    plot_shap_bar / plot_shap_beeswarm: summary plots for a target

    Inputs
    -----
    scaler: Scaler used for preprocessing - used to transform dataset
    model: the keras NN model used
    X_train_kmeans: the summarized dataset using kmeans
    X_test: numpy array BEFORE scaling - dataset to explain
    features: list of feature names used for training
    targets: list of model output names, in model output order.
             MUST have length == number of model outputs.
    categorical_features: dict mapping root feature to dummies
    features_dict: periodic features dict mapping root feature to periodic features
    ex_features: features that are non-periodic

    SHAP version handling
    ---------------------
    Older SHAP returned a list of (n_samples, n_features) arrays per output.
    Current SHAP returns one array of shape (n_samples, n_features, n_outputs).
    `_target_matrix` normalises both.
    """

    PERIODS = ['YEARH_', 'YEAR1_', 'YEAR1H_', 'YEAR2_', 'YEAR2H_', 'YEAR3_']

    def __init__(self, scaler, model, X_train_kmeans, X_test, features, features_dict,
                 targets, categorical_features, ex_features, full_targets=None):

        self.model = model
        self.X_test = X_test
        self.features = list(features)
        self.targets = list(targets)
        # kept for backwards compatibility; defaults to targets
        self.full_targets = list(full_targets) if full_targets is not None else list(targets)
        self.scaler = scaler
        self.categorical_features = categorical_features
        self.ex_features = ex_features
        self.train_df = X_train_kmeans   # kmeans summarized data
        self.time_features = features_dict
        self.X_scaled = scaler.transform(X_test)

    # ------------------------------------------------------------------
    # helpers
    # ------------------------------------------------------------------
    def _target_index(self, target):
        """Resolve a target name to its position on the model output axis."""
        if target in self.targets:
            return self.targets.index(target)
        if target in self.full_targets:
            return self.full_targets.index(target)
        raise KeyError(f"'{target}' not found in targets or full_targets.")

    def _target_matrix(self, sv, target_index):
        """Return (n_samples, n_features) for a single target, any SHAP version."""
        if isinstance(sv, list):                       # legacy SHAP
            arr = np.asarray(sv[target_index])
        else:
            sv = np.asarray(sv)
            if sv.ndim == 3:                           # (n_samples, n_features, n_outputs)
                arr = sv[..., target_index]
            elif sv.ndim == 2:                         # single-output model
                arr = sv
            else:
                raise ValueError(f"Unexpected SHAP array shape: {sv.shape}")

        arr = np.atleast_2d(arr)
        if arr.shape[1] != len(self.features):
            raw = len(sv) if isinstance(sv, list) else np.asarray(sv).shape
            raise ValueError(
                f"SHAP matrix for target index {target_index} has shape {arr.shape}; "
                f"expected (n_samples, {len(self.features)}). Raw SHAP output was {raw}. "
                f"Check that `targets` matches the model's output order and length."
            )
        return arr

    def _raw_values(self, n_rows):
        """Unscaled feature values for the first n_rows, for display / colouring."""
        return np.asarray(self.X_test)[:n_rows]

    # ------------------------------------------------------------------
    # single-patient explanation
    # ------------------------------------------------------------------
    def shap_explainer_generator(self, row_ind, sampling_times=10000):
        shap_explainer = shap.KernelExplainer(self.model, self.train_df, seed=random_state)

        self.row_ind = row_ind

        self.explain_instance = shap_explainer.shap_values(
            X=self.X_scaled[row_ind:row_ind + 1], n_samples=sampling_times
        )
        self.expected_value = shap_explainer.expected_value

        return self

    def shap_top_features(self, target, num_features=50):
        """Signed SHAP values for one patient. Direction is meaningful here."""
        target_index = self._target_index(target)
        arr = self._target_matrix(self.explain_instance, target_index)   # (1, n_features)

        df = pd.DataFrame({
            'Feature name': self.features,
            'SHAPley value': arr[0].reshape(-1,),
            'Value': np.asarray(self.X_test)[self.row_ind].reshape(-1,).round(decimals=2),
        })

        # display missing value as NA
        df['Value'] = df['Value'].replace(-1, 'NA')

        # aggregate categorical dummies back to the root variable
        for variable in self.categorical_features:
            dummies = self.categorical_features[variable]
            df_variable = df[df['Feature name'].isin(dummies)].reset_index(drop=True)
            if df_variable.empty:
                continue

            sum_relative_imp = df_variable['SHAPley value'].sum()
            shapley_values = df_variable['SHAPley value'].values

            try:
                ind = df_variable[df_variable['Value'] == 1].index
                feature_category = df_variable.iloc[ind[0]]['Feature name'].rsplit("_")[1]
            except (IndexError, KeyError):
                feature_category = 'Unknown'

            tmp = pd.DataFrame({'Feature name': variable,
                                'SHAPley value': sum_relative_imp,
                                'Value': feature_category,
                                'Indiv_SHAPley values': [shapley_values]}, index=[0])
            df = pd.concat([df, tmp], axis=0, ignore_index=True)
            df = df[~df['Feature name'].isin(dummies)]

        # aggregate periodic features back to the root variable
        for variable in self.time_features:
            members = self.time_features[variable]
            df_variable = df[df['Feature name'].isin(members)].reset_index(drop=True)
            if df_variable.empty:
                continue

            sum_relative_imp = df_variable['SHAPley value'].sum()
            shapley_values = df_variable['SHAPley value'].values

            # missing periods become None instead of raising IndexError
            values = []
            for period in self.PERIODS:
                match = df_variable.loc[df_variable['Feature name'] == period + variable, 'Value']
                values.append(match.values[0] if len(match) else None)

            tmp = pd.DataFrame({'Feature name': variable,
                                'SHAPley value': sum_relative_imp,
                                'Value': [values],
                                'Indiv_SHAPley values': [shapley_values]}, index=[0])
            df = pd.concat([df, tmp], axis=0, ignore_index=True)
            df = df[~df['Feature name'].isin(members)]

        df = df.sort_values(by=["SHAPley value"], ascending=False)[:num_features]
        return df.reset_index(drop=True)

    def plot_shap_force(self, target, row_ind, max_features=None,
                        save_path=None, close=False):
        """Force plot for one patient. Set max_features to keep the plot readable."""
        target_index = self._target_index(target)

        self.shap_explainer_generator(row_ind=row_ind)
        shap_df = self.shap_top_features(target=target, num_features=10 ** 6)

        shap_df['Abs_SHAP'] = np.abs(shap_df['SHAPley value'])
        shap_df = shap_df.sort_values('Abs_SHAP', ascending=False).reset_index(drop=True)

        plot_df = shap_df if max_features is None else shap_df.iloc[:max_features]
        plot_df = plot_df.copy()
        plot_df['FeatureNum'] = [f"Feature {i + 1}" for i in range(len(plot_df))]

        # expected_value may be scalar, list, or array depending on model / SHAP version
        base = np.atleast_1d(np.asarray(self.expected_value)).ravel()
        base_value = float(base[target_index]) if base.size > 1 else float(base[0])

        fig = plt.figure(figsize=(30, 6))
        shap.plots.force(base_value=base_value,
                         shap_values=plot_df['SHAPley value'].values,
                         feature_names=plot_df['FeatureNum'].values,
                         matplotlib=True,
                         show=False)

        if save_path:
            plt.savefig(save_path, bbox_inches='tight', dpi=300)
        if close:
            plt.close(fig)

        return fig, shap_df

    # ------------------------------------------------------------------
    # multi-sample explanation
    # ------------------------------------------------------------------
    def shap_explainer_generator_multiple(self, num_samples=50, sampling_times=500):
        num_samples = min(num_samples, len(self.X_test))
        test_df = self.X_scaled[:num_samples]
        self.n_explained = num_samples

        shap_explainer = shap.KernelExplainer(self.model, self.train_df, seed=random_state)

        self.explain_multiple = shap_explainer.shap_values(X=test_df, n_samples=sampling_times)

        shape_desc = (f"list of {len(self.explain_multiple)}"
                      if isinstance(self.explain_multiple, list)
                      else np.asarray(self.explain_multiple).shape)
        print(f"Generated SHAP explanation for {num_samples} samples, "
              f"re-evaluating {sampling_times} times. SHAP output: {shape_desc}")

        return self

    def shap_top_features_multiple(self, target, num_features=50, aggregate_time=True):
        """Mean |SHAP| per feature for one target, descending.

        abs is taken BEFORE summing across a feature group (v6 behaviour), so
        contributions in opposite directions across time windows do not cancel.
        Direction is therefore not recoverable from this output.

        Set aggregate_time=False to keep individual YEAR* features, e.g. if the
        downstream notebook does its own prefix-stripping and groupby.
        """
        target_index = self._target_index(target)
        arr = self._target_matrix(self.explain_multiple, target_index)

        df = pd.DataFrame({
            'Feature name': self.features,
            'SHAPley value': np.average(np.abs(arr), axis=0),
        })

        for variable in self.categorical_features:
            dummies = self.categorical_features[variable]
            df_variable = df[df['Feature name'].isin(dummies)]
            if df_variable.empty:
                continue
            df = pd.concat([df, pd.DataFrame({
                'Feature name': [variable],
                'SHAPley value': [df_variable['SHAPley value'].sum()]})], ignore_index=True)
            df = df[~df['Feature name'].isin(dummies)]

        if aggregate_time:
            for variable in self.time_features:
                members = self.time_features[variable]
                df_variable = df[df['Feature name'].isin(members)]
                if df_variable.empty:
                    continue
                df = pd.concat([df, pd.DataFrame({
                    'Feature name': [variable],
                    'SHAPley value': [df_variable['SHAPley value'].sum()]})], ignore_index=True)
                df = df[~df['Feature name'].isin(members)]

        df = df.sort_values(by=["SHAPley value"], ascending=False)
        return df[:num_features].reset_index(drop=True)

    # ------------------------------------------------------------------
    # aggregation for plots
    # ------------------------------------------------------------------
    def _aggregate_shap_values(self, shap_values, feature_values, abs_first=False):
        """Aggregate SHAP values for categorical and time features.

        abs_first=True  -> |SHAP| before summing a group (matches the table)
        abs_first=False -> signed sum, keeps direction (needed for beeswarm)
        """
        values = np.atleast_2d(np.asarray(shap_values))
        feature_values = np.atleast_2d(np.asarray(feature_values))

        if values.shape[1] != len(self.features):
            raise ValueError(f"SHAP matrix has {values.shape[1]} columns, "
                             f"expected {len(self.features)} features.")
        if feature_values.shape[0] != values.shape[0]:
            raise ValueError(f"{values.shape[0]} SHAP rows vs "
                             f"{feature_values.shape[0]} feature-value rows.")

        contrib = np.abs(values) if abs_first else values

        features = np.array(self.features)
        agg_values, agg_features, agg_feature_values = [], [], []
        mask = np.ones(len(features), dtype=bool)

        # categorical
        for variable in self.categorical_features:
            indices = [i for i, f in enumerate(features)
                       if f in self.categorical_features[variable]]
            if not indices:
                continue
            mask[indices] = False
            agg_values.append(np.sum(contrib[:, indices], axis=1))
            agg_features.append(variable)
            # which dummy is active -> ordinal code, used only for plot colouring
            agg_feature_values.append(
                np.argmax(feature_values[:, indices], axis=1).astype(float))

        # periodic
        for variable in self.time_features:
            indices = [i for i, f in enumerate(features)
                       if f in self.time_features[variable]]
            if not indices:
                continue
            mask[indices] = False
            agg_values.append(np.sum(contrib[:, indices], axis=1))
            agg_features.append(variable)
            agg_feature_values.append(np.mean(feature_values[:, indices], axis=1))

        # remaining
        for idx in np.where(mask)[0]:
            agg_values.append(contrib[:, idx])
            agg_features.append(features[idx])
            agg_feature_values.append(feature_values[:, idx])

        if not agg_values:
            return np.array([]), np.array([]), np.array([])

        return (np.stack(agg_values).T,
                np.array(agg_features),
                np.stack(agg_feature_values).T)

    def plot_shap_bar(self, target, max_display=20, exclude_features=None,
                      save_path=None, close=False):
        """Mean |SHAP| bar plot. Ranking matches shap_top_features_multiple."""
        target_index = self._target_index(target)
        arr = self._target_matrix(self.explain_multiple, target_index)

        agg_values, agg_features, _ = self._aggregate_shap_values(
            arr, self._raw_values(arr.shape[0]), abs_first=True)

        if exclude_features:
            keep = ~np.isin(agg_features, exclude_features)
            agg_values, agg_features = agg_values[:, keep], agg_features[keep]

        fig = plt.figure(figsize=(12, 6))
        shap.summary_plot(agg_values,
                          feature_names=list(agg_features),
                          plot_type="bar",
                          max_display=max_display,
                          show=False)
        plt.title(f'SHAP Feature Importance for {target}')
        plt.gca().xaxis.label.set_size(10)
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, bbox_inches='tight', dpi=300)
        if close:
            plt.close(fig)
        return fig

    def plot_shap_beeswarm(self, target, max_display=20, exclude_features=None,
                           save_path=None, close=False):
        """Beeswarm of signed SHAP values; colour = raw (unscaled) feature value."""
        target_index = self._target_index(target)
        arr = self._target_matrix(self.explain_multiple, target_index)

        agg_values, agg_features, agg_feature_values = self._aggregate_shap_values(
            arr, self._raw_values(arr.shape[0]), abs_first=False)

        if exclude_features:
            keep = ~np.isin(agg_features, exclude_features)
            agg_values = agg_values[:, keep]
            agg_features = agg_features[keep]
            agg_feature_values = agg_feature_values[:, keep]

        fig = plt.figure(figsize=(12, 8))
        shap.summary_plot(agg_values,
                          agg_feature_values,
                          feature_names=list(agg_features),
                          max_display=max_display,
                          show=False)
        plt.title(f'SHAP Feature Impact for {target}')
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, bbox_inches='tight', dpi=300)
        if close:
            plt.close(fig)
        return fig

## Import Data

Compare with original code, add train2019 to the train set because this is the fact

In [ ]:
# data files
feature_names = pd.read_csv(features_file)['Feature name'].tolist()
train = pd.read_csv(train_file, usecols=feature_to_drop+feature_names)
train_col = train.columns.tolist()

# Load prediction
train_wpreds = pd.read_csv(PROCDATAPATH + TRAINPREDSFILE)

# Load feature map
feature_map = pd.read_csv(DATAPATH3 + FEATUREMAPFILE)
feature_dict = dict(zip(feature_map['FEATURE_NAME'], feature_map['XAI_NAME']))
display_dict = dict(zip(feature_map['XAI_NAME'], feature_map['XAI_DISPLAY']))

# Load target and exclusion file
target_map = pd.read_csv(DATAPATH3 + TARGETMAPFILE)

del train
gc.collect()

## Load Modela and Scaler

In [ ]:
# model = keras.models.load_model(model_file, compile=False)
model = tf.keras.models.load_model(model_file, compile=False)
scaler = load(open(scaler_file, 'rb'))
train_kmeans = load(open(kmeans_file, 'rb'))
features_dict = load(open(featdict_file, 'rb'))

## Process Data

In [ ]:
# Define process function, combine scaler
def data_proc_X(data, feature_names, scaler_file=None):
    X = data[feature_names]

    # Added to fill in missing
    col_counts = [col for col in feature_names if col in count_cols]
    X[col_counts] = X[col_counts].fillna(0)  # Fill in missing counts with 0

    X.fillna(-1, inplace=True)  # Fill in missing with -1
    X = np.asarray(X)  # Convert to array

    if scaler_file is None:
        X = X
    else:
        scaler = load(open(scaler_file, 'rb'))  # Load scaler
        X = scaler.transform(X)  # Scale data
    return X

def data_proc_y(data, target_col):
    # Form np arrays of labels
    y = np.asarray(data[target_col])
    return y

## Helper Functions

In [ ]:
import warnings
import textwrap, matplotlib.pyplot as plt

def build_display_map(feature_map_path, non_periodic_features, categorical_features,
                      features_dict, key_col='FEATURE_NAME', display_col='XAI_DISPLAY'):
    """Map the labels that actually appear on a SHAP plot to their display names."""
    fmap = pd.read_csv(feature_map_path)

    # 1. Validation
    if key_col not in fmap.columns:
        raise KeyError(f"No feature-name column in {feature_map_path}. "
                        f"Columns: {list(fmap.columns)}. Pass key_col= explicitly.")
    if display_col not in fmap.columns:
        raise KeyError(f"'{display_col}' not in {feature_map_path}. Columns: {list(fmap.columns)}")

    # 2. Clean
    raw = (fmap[[key_col, display_col]].dropna()
           .astype(str).apply(lambda s: s.str.strip())
           .set_index(key_col)[display_col].to_dict())

    # 3. Display groups
    display_map, conflicts, missing = {}, {}, []

    for group_label, members in features_dict.items():          # periodic groups
        names = {raw[m] for m in members if m in raw}
        if not names:
            missing.append(group_label)
            continue
        if len(names) > 1:
            conflicts[group_label] = sorted(names)
        display_map[group_label] = sorted(names)[0]

    for block_label, members in (categorical_features or {}).items():   # one-hot blocks
        names = {raw[m] for m in members if m in raw}
        if len(names) > 1:
            conflicts[block_label] = sorted(names)
        display_map[block_label] = sorted(names)[0] if names else block_label
        if not names:
            missing.append(block_label)

    collapsed = {m for members in (categorical_features or {}).values() for m in members}
    for f in non_periodic_features:                              # everything else
        if f in collapsed:
            continue
        if f in raw:
            display_map[f] = raw[f]
        else:
            missing.append(f)

    print(f"display map: {len(display_map)} plot labels resolved")
    if conflicts:
        warnings.warn(f"{len(conflicts)} label(s) map to >1 XAI_DISPLAY value; "
                      f"first alphabetically used: {conflicts}")
    if missing:
        print(f"  {len(missing)} label(s) not in the map, original names kept: {missing}")
    return display_map



def wrap_label(text, width=30, max_lines=2):
    """Wrap a long tick label onto balanced lines, never truncating.

    Finds the narrowest width that fits the text in n lines, for the smallest n that
    works, which balances the lines instead of leaving a long first line and a stub.
    If the text cannot fit max_lines at the requested width, the width is relaxed
    rather than the text being cut.
    """
    text = ' '.join(str(text).split())
    if len(text) <= width:
        return text
    longest = max(len(w) for w in text.split())
    for n in range(2, max_lines + 1):
        for w in range(max(longest, len(text) // n), width + 1):
            lines = textwrap.wrap(text, w)
            if len(lines) <= n:
                return '\n'.join(lines)
    w = width
    while len(textwrap.wrap(text, w)) > max_lines:
        w += 1
    return '\n'.join(textwrap.wrap(text, w))


def relabel_plot(display_map, ax=None, width=30, max_lines=2, fontsize=10):
    """Rewrite y-tick labels using display_map, wrapping long ones onto up to max_lines."""
    ax = plt.gca() if ax is None else ax
    labels = [wrap_label(display_map.get(t.get_text().strip(), t.get_text()), width, max_lines)
              for t in ax.get_yticklabels()]
    ax.set_yticks(ax.get_yticks())
    ax.set_yticklabels(labels, fontsize=fontsize, linespacing=0.95)
    ax.tick_params(axis='y', length=0)
    plt.tight_layout()

## Analyze Top Positive Patients

In [ ]:
CATEGORICAL_FEATURES = {
    'GENDER': ['GENDER_Male', 'GENDER_Female'],
    'RACE': ['RACE_Chinese', 'RACE_Malay', 'RACE_Indian', 'RACE_Others'],
}

display_map = build_display_map(
    f'{DATAPATH3}ACE_Feature_Map.csv',      # adjust path
    non_periodic_features, CATEGORICAL_FEATURES, features_dict,
)

In [ ]:
bar_plots = []
beeswarm_plots = []

for target in target_col[:-1]:
    print(target)
    df = train_wpreds[(train_wpreds[target+'_Prior']==0)&(train_wpreds[target]==1)].sort_values(
        target+'_Proba', ascending=False).head(1000)
    df = df[train_col].copy()

    # Instantiate the DeepPatientShap class --> Only need to instantiate once
    shap_explain = DeepPatientShap(
        scaler=scaler,
        model=model,
        X_train_kmeans=train_kmeans,  # summarized dataset
        X_test=data_proc_X(df, feature_names),  # dataset to generate feature importance for (ie production data)
        features=feature_names,
        targets=target_col,
        categorical_features={
            'GENDER': ['GENDER_Male', 'GENDER_Female'],
            'RACE': ['RACE_Chinese', 'RACE_Malay', 'RACE_Indian', 'RACE_Others'],
        },
        ex_features=non_periodic_features,
        features_dict=features_dict,
        )
    shap_explain.shap_explainer_generator_multiple(num_samples=100)

    # Save Feature importance based on individual absolute values: consistent with bar plot
    feat_df = shap_explain.shap_top_features_multiple(target, num_features=1000)
    feat_df.to_csv(f'{SUBDATAPATH2}Model{model_name}_SHAP_imp_TP_{target}_2026.csv', index=False)

    # Save feature importance based on sum shapley values: consistent with beeswarm
    ti = shap_explain._target_index(target)
    arr = shap_explain._target_matrix(shap_explain.explain_multiple, ti)
    raw = shap_explain._raw_values(arr.shape[0])
    sgn_vals, sgn_feats, _ = shap_explain._aggregate_shap_values(arr, raw, abs_first=False)

    feat_signed_df = pd.DataFrame({
        'Feature name': sgn_feats,
        'mean_SHAP': sgn_vals.mean(axis=0),
        'mean_abs_SHAP': np.abs(sgn_vals).mean(axis=0),
        'sd_SHAP': sgn_vals.std(axis=0),
        'pct_positive': (sgn_vals > 0).mean(axis=0),
    })
    feat_signed_df = feat_signed_df.sort_values(
        'mean_abs_SHAP', ascending=False).reset_index(drop=True)
    feat_signed_df.to_csv(
        f'{SUBDATAPATH2}Model{model_name}_SHAP_signed_TP_{target}_2026.csv', index=False)

    # Create and display bar plot
    bar_plot = shap_explain.plot_shap_bar(target=target, max_display=10)
    relabel_plot(display_map)
    plt.savefig(f"{FIGPATH}barplot_{target}.png", dpi=300, bbox_inches='tight')
    bar_plots.append(bar_plot)
    plt.show()

    # Create and display beeswarm plot
    beeswarm_plot = shap_explain.plot_shap_beeswarm(target=target, max_display=10)
    relabel_plot(display_map)
    plt.savefig(f"{FIGPATH}beeswarm_{target}.png", dpi=300, bbox_inches='tight')
    beeswarm_plots.append(beeswarm_plot)
    plt.show()

    print(target, "saved")


# End